# 📊 SDMX Agent — querying AfDB data in plain language
### STG17 Workshop · AfDB / STATAFRIC · Advanced lab
### ✅ Version 7 — 24 September 2026 · full catalog · strict JSON mode · tolerant of “reasoning” models

> **In one sentence:** you will build an agent that **discovers** a SDMX service's datasets on its own, **builds its own tools** from what it finds, **plans** the calls needed to answer a question asked in plain English, then **displays** the result as a table and a chart — with a full trace of every request.

| ⏱ Duration | 🌍 Target service | 🆓 Cost | 🔑 API key | 🧩 Dependencies |
|:--:|:--:|:--:|:--:|:--:|
| 2 h | SDMX REST (.Stat / NSI) | free | optional | `requests`, `pandas`, `matplotlib`, `ipywidgets` |

---

## 🎯 The two target services

| Service | Role | URL |
|:--|:--|:--|
| **NSI Web Service** (AfDB) | the **SDMX engine**: structures and data | `https://datamanager.afdb.org/ws/nsi_ws/rest/` |
| **Data Browser** (AfDB) | the **web front-end** consuming that engine | `https://dbrtest.opendataforafrica.org/#/en/afdb_test` |

The Data Browser is a web application: it cannot be queried from Python. The agent therefore targets the **NSI Web Service** — the very service the browser itself uses.

> ⚠️ **To check on first run.** The service home page states that the instance may only be reachable **from the AfDB network** (“Is usable from external users: False”). The notebook tests the connection at start-up and, if the service is unreachable, falls back to:
> 1. a **public SDMX service** (same .Stat/NSI technology), so the workshop still runs;
> 2. failing that, a **demonstration snapshot** embedded in the notebook.
>
> The mode in use is displayed in plain sight, and the agent always states **where its figures come from**.

---

## 🗺️ Roadmap

| Part | Step | Content |
|:--|:--:|:--|
| **🟦 Understand** | 1 | Setup and configuration |
| | 2 | SDMX in 5 minutes: dataflow, DSD, dimensions, codelists, key |
| **🟩 Connect** | 3 | The SDMX client: requests, cache, errors, fallbacks |
| | 4 | Automatic discovery of the **whole catalog** |
| **🟨 Equip** | 5 | The agent's tools, **generated from the service** |
| | 6 | The protocol and **planning** |
| | 7 | Policy, budget, audit log |
| **🟪 Think** | 8 | The brains: local planner or free model |
| | 9 | Text-mode trial |
| **🟥 Use** | 10 | **The plain-language interface** |
| | 11 | Under the hood |
| **⬛ Wrap up** | 12 | Limits, exercises, troubleshooting |

---
# 🟦 PART 1 — Understand

## Step 1 · Setup and configuration

**🎯 Goal:** the same environment for everyone, and one central place to configure the target service.

Only one variable matters: `ENDPOINTS`. It lists, in order of preference, the SDMX services to try. The agent uses the first one that answers.

In [1]:
# ── Step 1 · Setup and configuration ────────────────────────────────────────
import importlib, subprocess, sys

for module, paquet in [("requests", "requests"), ("pandas", "pandas>=2.0"),
                       ("matplotlib", "matplotlib>=3.7"), ("ipywidgets", "ipywidgets>=7.6")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquet], check=False)

import ast, base64, hashlib, html, io, json, operator, os, re, time, unicodedata, warnings
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, Optional

import matplotlib
matplotlib.use("Agg")                      # figures are rendered as images, not shown directly
import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import HTML, display

# ── SDMX services, in order of preference ───────────────────────────────────
ENDPOINTS = [
    {"name": "AfDB · NSI Web Service", "url": "https://datamanager.afdb.org/ws/nsi_ws/rest",
     "agency": "all", "note": "the workshop target; may be restricted to AfDB's internal network"},
    {"name": ".Stat · public demonstration service", "url": "https://nsi-demo-stable.siscc.org/rest",
     "agency": "all", "note": "same technology (.Stat/NSI), publicly reachable"},
    {"name": "Pacific Data Hub · .Stat", "url": "https://stats-nsi-stable.pacificdata.org/rest",
     "agency": "SPC", "note": "another public .Stat instance, rich in dataflows"},
]
LANGUAGE = "en"                 # language requested from the service (Accept-Language header)
TIMEOUT = 25                    # seconds before a request is abandoned
MAX_OBSERVATIONS = 5000       # guard rail: beyond this, the request is refused

ROOT = Path("stg17_sdmx")
CACHE = ROOT / "cache"
OUTPUTS = ROOT / "sorties"
for d in (CACHE, OUTPUTS):
    d.mkdir(parents=True, exist_ok=True)

# ── Charte graphique et fonctions d’affichage ───────────────────────────────
GREEN, NAVY, GOLD, RED, GREY, LIGHT = "#1B7A43", "#0B2545", "#F2A900", "#B83B2E", "#6B7B75", "#EAF5EE"
FONT = "font-family:Segoe UI,system-ui,-apple-system,sans-serif"
plt.rcParams.update({"figure.dpi": 120, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#9AA8A1", "axes.titleweight": "bold", "axes.titlesize": 12})

def _norm(text):
    """Lower-case, accent-free: to compare labels regardless of spelling."""
    text = unicodedata.normalize("NFKD", str(text).lower())
    return "".join(c for c in text if not unicodedata.combining(c))


_STYLES = {"info": (GREEN, "#EAF5EE", "💡"), "key": (NAVY, "#E8EEF6", "🔑"), "warning": (GOLD, "#FFF7E0", "⚠️"),
           "danger": (RED, "#FBECEA", "⛔"), "success": (GREEN, "#E3F4E9", "✅")}

def callout(text, title="", kind="info"):
    colour, background, icon = _STYLES[kind]
    display(HTML(f"<div style='border-left:5px solid {colour};background:{background};padding:11px 16px;border-radius:8px;"
                 f"margin:8px 0;{FONT};color:#1d2b24;line-height:1.5'><b>{icon} {title}</b>"
                 f"<div style='margin-top:3px'>{text}</div></div>"))

def df_to_html(df, max_rows=15):
    preview = df.head(max_rows)
    headers = "".join(f"<th style='background:#0B2545;color:#fff;padding:5px 9px;text-align:left'>{html.escape(str(c))}</th>"
                      for c in preview.columns)
    body = ""
    for i, (_, row) in enumerate(preview.iterrows()):
        background = "#F4F7F5" if i % 2 else "#FFFFFF"
        body += "<tr>" + "".join(f"<td style='padding:5px 9px;border-bottom:1px solid #DDE3E0;background:{background}'>"
                                  f"{html.escape(str(v))}</td>" for v in row.values) + "</tr>"
    reste = (f"<div style='color:{GREY};font-size:11.5px;margin-top:3px'>… {len(df) - max_rows} row(s) de plus</div>"
             if len(df) > max_rows else "")
    return (f"<div style='overflow-x:auto'><table style='border-collapse:collapse;font-size:12px;{FONT};color:#1d2b24'>"
            f"<tr>{headers}</tr>{body}</table></div>{reste}")

def show_table(df, caption=None, max_rows=15):
    bloc = df_to_html(df, max_rows)
    if caption:
        bloc += f"<div style='color:{GREY};font-size:12px;font-style:italic;margin:2px 0 10px'>{caption}</div>"
    display(HTML(bloc))

def banner(overline, title, subtitle):
    display(HTML(f"<div style='background:linear-gradient(135deg,#0B2545 0%,#12507A 50%,#1B7A43 100%);border-radius:16px;"
                 f"padding:22px 30px;{FONT};margin:6px 0'>"
                 f"<div style='color:#F2A900;font-size:12px;letter-spacing:3px;font-weight:700'>{overline}</div>"
                 f"<div style='color:#fff;font-size:26px;font-weight:800;margin:6px 0'>{title}</div>"
                 f"<div style='color:#dbe7e0;font-size:14.5px;line-height:1.5'>{subtitle}</div></div>"))

VERSION = "v7 · robust planning"
banner(f"STG17 WORKSHOP · AfDB / STATAFRIC · {VERSION}", "📊 SDMX Agent",
        "A question in plain English → a plan → SDMX requests → a table and a chart, "
        "<b style='color:#fff'>with a trace of every call</b>.")
callout(f"Working folder : <code>{ROOT}/</code> · request cache : <code>{CACHE}/</code>",
        "Environment ready", "success")

## Step 2 · SDMX in 5 minutes

**🎯 Goal:** understand the four notions without which no agent can query a SDMX service.

**SDMX** (*Statistical Data and Metadata eXchange*) is ISO standard 17369 for exchanging statistical data. A SDMX service does not expose “tables”: it exposes **cubes** described by their metadata.

| Notion | What it is | Example |
|:--|:--|:--|
| **Dataflow** | a queryable dataset | `DF_POP` — Population |
| **DSD** (*Data Structure Definition*) | the cube's structure: which dimensions, in which order | `FREQ · REF_AREA · INDICATOR · TIME_PERIOD` |
| **Dimension** | one axis of the cube | `REF_AREA` (country) |
| **Codelist** | the allowed values of a dimension | `CL_AREA` → `CIV` = Côte d'Ivoire |
| **Key** | the selection inside the cube, dimensions separated by `.` | `A.CIV.GDP_GROWTH` |

**A data request URL** always follows the same pattern:

```
{service}/data/{agency},{dataflow},{version}/{key}?startPeriod=2015&endPeriod=2023
```

An empty slot **widens** the selection, a `+` **adds** one:

| Key | Meaning |
|:--|:--|
| `A.CIV.GDP_GROWTH` | annual, Côte d'Ivoire, GDP growth |
| `A.CIV+SEN+GHA.GDP_GROWTH` | three countries at once |
| `A..GDP_GROWTH` | every country (2nd dimension left empty) |
| `all` | the whole dataflow (⚠️ often very large) |

**🧠 Why this matters for an agent.** The service **describes itself**: the agent can ask for the list of dataflows, then the structure of the one it needs, then the allowed codes. It therefore needs no hard-coded list of countries or indicators: **it builds its own tools from the metadata**.

In [2]:
# ── Step 2 · Diagram: from the service to the answer ────────────────────────
def _box(x, y, w, h, title, sub, border, background="#FFFFFF"):
    return (f"<rect x='{x}' y='{y}' width='{w}' height='{h}' rx='11' fill='{background}' stroke='{border}' stroke-width='2'/>"
            f"<text x='{x + w/2}' y='{y + h/2 - 3}' text-anchor='middle' font-size='13' font-weight='700' fill='#1d2b24'>{title}</text>"
            f"<text x='{x + w/2}' y='{y + h/2 + 15}' text-anchor='middle' font-size='10.5' fill='#56655e'>{sub}</text>")

def _arrow(x1, y1, x2, y2, coul="#56655e", label_text=""):
    return (f"<line x1='{x1}' y1='{y1}' x2='{x2}' y2='{y2}' stroke='{coul}' stroke-width='2' marker-end='url(#fl)'/>"
            + (f"<text x='{(x1+x2)/2}' y='{(y1+y2)/2 - 7}' text-anchor='middle' font-size='10.5' fill='{coul}'"
               f" font-weight='600'>{label_text}</text>" if label_text else ""))

display(HTML(f"""<svg viewBox='0 0 950 230' width='100%' style='max-width:950px;background:#fff;{FONT}'>
<defs><marker id='fl' markerUnits='userSpaceOnUse' markerWidth='11' markerHeight='11' refX='10' refY='5.5' orient='auto'>
<path d='M0,0 L11,5.5 L0,11 z' fill='#56655e'/></marker></defs>
{_box(10, 80, 150, 62, "❓ Question", "in plain English", NAVY, "#E8EEF6")}
{_box(200, 80, 160, 62, "🗺️ Plan", "which calls, in which order", GOLD, "#FFF7E0")}
{_box(400, 20, 170, 62, "📚 /dataflow", "which datasets?", GREEN, LIGHT)}
{_box(400, 100, 170, 62, "🧱 /datastructure", "dimensions & codelists", GREEN, LIGHT)}
{_box(400, 175, 170, 45, "📈 /data", "the observations", GREEN, LIGHT)}
{_box(620, 80, 150, 62, "🧮 Traitement", "pandas : filtre, calcul", NAVY, "#E8EEF6")}
{_box(800, 80, 140, 62, "📊 Answer", "table + chart", GREEN, LIGHT)}
{_arrow(160, 111, 198, 111)}
{_arrow(360, 100, 398, 55, "#56655e", "1")}
{_arrow(360, 111, 398, 128, "#56655e", "2")}
{_arrow(360, 122, 398, 190, "#56655e", "3")}
{_arrow(572, 55, 618, 100)}
{_arrow(572, 131, 618, 115)}
{_arrow(572, 195, 618, 125)}
{_arrow(770, 111, 798, 111)}
</svg>"""))
callout("The agent never guesses a country code or an indicator: it first queries the <b>metadata</b> "
        "(steps 1 and 2), then builds the key of the data request (step 3).", "The principle", "key")

---
# 🟩 PART 2 — Connect

## Step 3 · The SDMX client

**🎯 Goal:** a **robust** network layer, because an agent talking to a remote service must survive slowness, unexpected formats and outages.

The client does six things a plain `requests.get` does not:

1. **Negotiates the format**: it asks for SDMX-JSON and falls back to `?format=sdmx-json` if the service rejects the `Accept` header;
2. **Caches** every response on disk: the same request is never paid for twice (and the workshop stays fast);
3. **Rate-limits** itself so as not to overload the service;
4. **Translates HTTP errors** into messages the agent can read (404 = unknown dataflow, 413 = request too large…);
5. **Understands both versions** of SDMX-JSON (1.0 and 2.0), whose structures differ;
6. **Falls back** from one service to the next, then to the demonstration snapshot.

In [3]:
# ── Step 3 · SDMX REST client ───────────────────────────────────────────────
ACCEPT_STRUCTURE = "application/vnd.sdmx.structure+json;version=1.0.0"
ACCEPT_DATA = "application/vnd.sdmx.data+json;version=1.0.0"


class SDMXError(Exception):
    """Error returned by the service, translated into a readable message."""


class SDMXClient:
    """SDMX REST 1.5 client: structures and data, with a disk cache and fallbacks."""

    def __init__(self, base_url, language=LANGUAGE, timeout=TIMEOUT, cache=CACHE, interval=0.4):
        self.base = base_url.rstrip("/")
        self.language, self.timeout, self.cache, self.interval = language, timeout, Path(cache), interval
        self.cache.mkdir(parents=True, exist_ok=True)
        self.requests_count, self.from_cache, self._dernier = 0, 0, 0.0
        self.http_log = []

    # ── network layer ──────────────────────────────────────────────────────
    def _cache_file(self, url):
        return self.cache / (hashlib.sha256(url.encode()).hexdigest()[:24] + ".json")

    def get(self, path, params=None, accept=ACCEPT_STRUCTURE, use_cache=True):
        url = f"{self.base}/{path.lstrip('/')}"
        if params:
            url += "?" + "&".join(f"{k}={v}" for k, v in params.items() if v not in (None, ""))
        fichier = self._cache_file(url)
        if use_cache and fichier.exists():
            self.from_cache += 1
            self.http_log.append({"url": url, "status": "cache", "ms": 0})
            return json.loads(fichier.read_text(encoding="utf-8"))

        waiting = self.interval - (time.monotonic() - self._dernier)
        if waiting > 0:
            time.sleep(waiting)
        self._dernier = time.monotonic()
        t0 = time.perf_counter()
        self.requests_count += 1
        headers = {"Accept": accept, "Accept-Language": self.language}
        try:
            r = requests.get(url, headers=headers, timeout=self.timeout)
            if r.status_code == 406:                        # format refused: try another way
                separateur = "&" if "?" in url else "?"
                r = requests.get(url + separateur + "format=sdmx-json", timeout=self.timeout,
                                 headers={"Accept-Language": self.language})
        except requests.RequestException as e:
            self.http_log.append({"url": url, "status": f"network: {type(e).__name__}", "ms": 0})
            raise SDMXError(f"service unreachable ({type(e).__name__})") from e
        duree = (time.perf_counter() - t0) * 1000
        self.http_log.append({"url": url, "status": r.status_code, "ms": round(duree)})

        if r.status_code == 404:
            raise SDMXError("not found (404): check the dataflow identifier, the agency or the key")
        if r.status_code == 413:
            raise SDMXError("request too large (413): narrow the key or the period")
        if r.status_code == 400:
            raise SDMXError(f"invalid request (400): {r.text[:150]}")
        if not r.ok:
            raise SDMXError(f"error HTTP {r.status_code}")
        if not r.text.strip():
            raise SDMXError("empty response (no data for this selection)")
        try:
            data = r.json()
        except ValueError as e:
            raise SDMXError("unreadable response (the service did not return JSON)") from e
        if use_cache:
            fichier.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
        return data

    # ── tolerant reading of both SDMX-JSON versions ────────────────────────
    @staticmethod
    def _root(payload):
        return payload.get("data", payload)

    @staticmethod
    def _label(obj, language=LANGUAGE):
        noms = obj.get("names") or {}
        name = noms.get(language) or obj.get("name") or noms.get("en")
        if isinstance(name, dict):
            name = name.get(language) or name.get("en") or next(iter(name.values()), "")
        return name or obj.get("id", "")

    # ── structures ─────────────────────────────────────────────────────────
    def dataflows(self, agency="all", limite=None):
        """List of dataflows: identifier, agency, version, name."""
        payload = self.get(f"dataflow/{agency}/all/latest", {"detail": "allstubs"})
        rows = [{"agency": f.get("agencyID", ""), "flow": f.get("id", ""), "version": f.get("version", "1.0"),
                   "name": self._label(f)} for f in self._root(payload).get("dataflows", [])]
        table = pd.DataFrame(rows).sort_values("flow").reset_index(drop=True)
        return table.head(limite) if limite else table

    def structure(self, agency, flow, version="latest"):
        """Dimensions of a dataflow and allowed codes (partial codelists)."""
        payload = self.get(f"dataflow/{agency}/{flow}/{version}",
                          {"references": "all", "detail": "referencepartial"})
        root = self._root(payload)
        codelists = {}
        for cl in root.get("codelists", []):
            urn = f"{cl.get('agencyID')}:{cl.get('id')}({cl.get('version', '1.0')})"
            codelists[urn] = [{"code": c.get("id"), "label": self._label(c)} for c in cl.get("codes", [])]
        dsd = (root.get("dataStructures") or [{}])[0]
        composants = dsd.get("dataStructureComponents", {})
        dimensions = []
        for dim in composants.get("dimensionList", {}).get("dimensions", []):
            enumeration = ((dim.get("localRepresentation") or {}).get("enumeration") or "")
            cle_cl = enumeration.split("=")[-1] if "=" in enumeration else ""
            dimensions.append({"id": dim.get("id"), "position": dim.get("position", len(dimensions) + 1),
                               "name": self._label(dim) if dim.get("name") else dim.get("id"),
                               "codes": codelists.get(cle_cl, [])})
        dimensions.sort(key=lambda d: d["position"])
        temps = composants.get("dimensionList", {}).get("timeDimensions", [])
        flow_name = next((self._label(f) for f in root.get("dataflows", []) if f.get("id") == flow), flow)
        return {"agency": agency, "flow": flow, "version": version, "name": flow_name,
                "dimensions": dimensions, "dimension_temps": (temps[0].get("id") if temps else "TIME_PERIOD"),
                "dsd": dsd.get("id", "")}

    # ── data ───────────────────────────────────────────────────────────────
    def data(self, agency, flow, key="all", version="latest", start=None, end=None, last_n=None):
        params = {"startPeriod": start, "endPeriod": end, "lastNObservations": last_n,
                  "dimensionAtObservation": "AllDimensions"}
        payload = self.get(f"data/{agency},{flow},{version}/{key or 'all'}", params, accept=ACCEPT_DATA)
        return self.to_dataframe(payload)

    @classmethod
    def to_dataframe(cls, payload):
        """Convertit un message SDMX-JSON (1.0 ou 2.0) en show_table plat."""
        root = cls._root(payload)
        datasets = root.get("dataSets") or []
        structures = root.get("structures") or ([root["structure"]] if "structure" in root else [])
        if not datasets or not structures:
            return pd.DataFrame()
        structure = structures[0]
        obs_dims = (structure.get("dimensions", {}).get("observation") or [])
        series_dims = (structure.get("dimensions", {}).get("series") or [])
        rows = []
        for dataset in datasets:
            # cases 1 : vue plate (AllDimensions)
            for key, values in (dataset.get("observations") or {}).items():
                indices = [int(i) for i in key.split(":")]
                row = {d["id"]: d["values"][i]["id"] for d, i in zip(obs_dims, indices)}
                row["OBS_VALUE"] = values[0] if values else None
                rows.append(row)
            # case 2: series view + observations
            for series_key, series in (dataset.get("series") or {}).items():
                indices = [int(i) for i in series_key.split(":")]
                base = {d["id"]: d["values"][i]["id"] for d, i in zip(series_dims, indices)}
                for obs_key, values in (series.get("observations") or {}).items():
                    row = dict(base)
                    for d, i in zip(obs_dims, [int(x) for x in obs_key.split(":")]):
                        row[d["id"]] = d["values"][i]["id"]
                    row["OBS_VALUE"] = values[0] if values else None
                    rows.append(row)
        table = pd.DataFrame(rows)
        if not table.empty:
            table["OBS_VALUE"] = pd.to_numeric(table["OBS_VALUE"], errors="coerce")
            # readable labels for each dimension
            for dim in list(obs_dims) + list(series_dims):
                labels = {v["id"]: cls._label(v) for v in dim.get("values", [])}
                if dim["id"] in table.columns and labels:
                    table[dim["id"] + "_LIBELLE"] = table[dim["id"]].map(labels)
            time_columns = [c for c in table.columns if c.startswith("TIME")]
            if time_columns:
                table = table.sort_values(time_columns[0]).reset_index(drop=True)
        return table

In [4]:
# ── Step 3 (cont.) · Multi-dataflow demonstration snapshot ──────────────────
# FICTIONAL data, in SDMX-JSON format, so the notebook runs even without a network.
_COUNTRIES = [("CIV", "Cote d'Ivoire"), ("SEN", "Senegal"), ("GHA", "Ghana"), ("NGA", "Nigeria"),
         ("TUN", "Tunisia"), ("MAR", "Morocco"), ("KEN", "Kenya")]
_YEARS = [str(a) for a in range(2015, 2024)]

DEMO_FLOWS = {
    "DF_MACRO_DEMO": {"name": "Macroeconomic indicators (demonstration)",
                      "indicateurs": [("GDP_GROWTH", "Real GDP growth", 5.0), ("INFLATION", "Inflation, consumer prices", 3.0),
                                      ("UNEMP", "Unemployment rate", 9.0), ("GDP_CAPITA", "GDP per capita, current USD", 2400.0)]},
    "DF_POP_DEMO": {"name": "Population and demography (demonstration)",
                    "indicateurs": [("POP_TOT", "Total population, millions", 28.0), ("URBAN_PCT", "Urban population, % of total", 48.0),
                                    ("POP_GROWTH", "Annual population growth", 2.6)]},
    "DF_EDU_DEMO": {"name": "Education (demonstration)",
                    "indicateurs": [("LITERACY", "Adult literacy rate", 62.0),
                                    ("COMPLETION_PRIM", "Primary completion rate", 78.0),
                                    ("PUPIL_TEACHER", "Pupil / teacher ratio", 38.0)]},
    "DF_TRADE_DEMO": {"name": "External trade and commodities (demonstration)",
                      "indicateurs": [("EXPORTS_GOODS", "Goods exports, % of GDP", 24.0),
                                      ("IMPORTS_GOODS", "Goods imports, % of GDP", 26.0),
                                      ("COCOA_PROD", "Cocoa production, thousand tonnes", 900.0)]},
    "DF_HEALTH_DEMO": {"name": "Health (demonstration)",
                      "indicateurs": [("LIFE_EXP", "Life expectancy at birth", 62.0),
                                      ("VACC_DTP3", "DTP3 immunisation coverage", 82.0)]},
}

def _demo_value(flow, country, indicateur, annee, base):
    """Fictional but deterministic series: only there to make the demo work."""
    decalage = (sum(ord(c) for c in country + indicateur) % 9) - 4
    choc = -0.25 * base if (annee == "2020" and indicateur in ("GDP_GROWTH", "EXPORTS_GOODS")) else 0.0
    tendance = (int(annee) - 2015) * (base * 0.012 if indicateur not in ("UNEMP", "PUPIL_TEACHER") else -base * 0.008)
    return round(base * (1 + decalage * 0.02) + tendance + choc, 1)

def _demo_structure(flow):
    conf = DEMO_FLOWS[flow]
    return {"data": {
        "dataflows": [{"id": flow, "agencyID": "DEMO", "version": "1.0", "name": conf["name"]}],
        "dataStructures": [{"id": f"DSD_{flow}", "dataStructureComponents": {"dimensionList": {
            "dimensions": [
                {"id": "FREQ", "position": 1, "localRepresentation": {"enumeration": "urn:…Codelist=DEMO:CL_FREQ(1.0)"}},
                {"id": "REF_AREA", "position": 2, "localRepresentation": {"enumeration": "urn:…Codelist=DEMO:CL_AREA(1.0)"}},
                {"id": "INDICATOR", "position": 3, "localRepresentation": {"enumeration": f"urn:…Codelist=DEMO:CL_IND_{flow}(1.0)"}}],
            "timeDimensions": [{"id": "TIME_PERIOD"}]}}}],
        "codelists": [
            {"id": "CL_FREQ", "agencyID": "DEMO", "version": "1.0", "codes": [{"id": "A", "name": "Annual"}]},
            {"id": "CL_AREA", "agencyID": "DEMO", "version": "1.0", "codes": [{"id": c, "name": n} for c, n in _COUNTRIES]},
            {"id": f"CL_IND_{flow}", "agencyID": "DEMO", "version": "1.0",
             "codes": [{"id": c, "name": n} for c, n, _ in conf["indicateurs"]]}]}}

def _demo_data(flow):
    conf = DEMO_FLOWS[flow]
    observations = {}
    for i, (country, _) in enumerate(_COUNTRIES):
        for j, (code, _, base) in enumerate(conf["indicateurs"]):
            for k, annee in enumerate(_YEARS):
                observations[f"0:{i}:{j}:{k}"] = [_demo_value(flow, country, code, annee, base)]
    return {"data": {"dataSets": [{"observations": observations}], "structure": {"dimensions": {"observation": [
        {"id": "FREQ", "values": [{"id": "A", "name": "Annual"}]},
        {"id": "REF_AREA", "values": [{"id": c, "name": n} for c, n in _COUNTRIES]},
        {"id": "INDICATOR", "values": [{"id": c, "name": n} for c, n, _ in conf["indicateurs"]]},
        {"id": "TIME_PERIOD", "values": [{"id": a, "name": a} for a in _YEARS]}]}}}}

SNAPSHOT_FLOWS = {"data": {"dataflows": [{"id": f, "agencyID": "DEMO", "version": "1.0", "name": c["name"]}
                                        for f, c in DEMO_FLOWS.items()]}}


class DemoClient(SDMXClient):
    """Offline client: replays a 5-dataflow snapshot. Figures are FICTIONAL."""

    def __init__(self):
        super().__init__("https://demo.local/rest")
        self.demo = True

    def get(self, path, params=None, accept=None, use_cache=True):
        self.requests_count += 1
        self.http_log.append({"url": f"[demo] {path}", "status": "demo", "ms": 0})
        if path.startswith("dataflow") and "all/latest" not in path:
            flow = path.split("/")[2]
            if flow not in DEMO_FLOWS:
                raise SDMXError(f"unknown dataflow in the snapshot: {flow}")
            return _demo_structure(flow)
        if path.startswith("dataflow"):
            return SNAPSHOT_FLOWS
        if path.startswith("data/"):
            reference, _, key = path[len("data/"):].partition("/")
            flow = reference.split(",")[1] if "," in reference else reference
            table = self.to_dataframe(_demo_data(flow))
            key = key.split("?")[0]
            if key and key != "all":
                for position, selection in enumerate(key.split(".")):
                    if not selection:
                        continue
                    column = ["FREQ", "REF_AREA", "INDICATOR"][position]
                    table = table[table[column].isin(selection.split("+"))]
            start, end = (params or {}).get("startPeriod"), (params or {}).get("endPeriod")
            if start:
                table = table[table["TIME_PERIOD"] >= str(start)]
            if end:
                table = table[table["TIME_PERIOD"] <= str(end)]
            return {"__dataframe__": table.reset_index(drop=True)}
        raise SDMXError(f"resource not simulated: {path}")

    def data(self, agency, flow, key="all", version="latest", start=None, end=None, last_n=None):
        payload = self.get(f"data/{agency},{flow},{version}/{key or 'all'}",
                          {"startPeriod": start, "endPeriod": end})
        table = payload["__dataframe__"]
        if last_n:
            groupes = [c for c in ("FREQ", "REF_AREA", "INDICATOR") if c in table.columns]
            table = table.sort_values("TIME_PERIOD").groupby(groupes, as_index=False).tail(int(last_n))
        return table.reset_index(drop=True)


def connect(endpoints=ENDPOINTS, forcer_demo=False):
    """Tries each service in order; falls back to the snapshot as a last resort."""
    if forcer_demo:
        return DemoClient(), {"name": "Demonstration snapshot", "url": "—", "agency": "DEMO",
                              "note": "no network connection; figures are fictional"}
    for endpoint in endpoints:
        client = SDMXClient(endpoint["url"])
        try:
            flow = client.dataflows(endpoint.get("agency", "all"), limite=1)
            if not flow.empty:
                return client, endpoint
        except Exception as e:
            print(f"   ⚠️ {endpoint['name']} : {e}")
    print("   ⚠️ no service reachable: falling back to the demonstration snapshot")
    return DemoClient(), {"name": "Demonstration snapshot", "url": "—", "agency": "DEMO",
                          "note": "no service reachable; figures are fictional"}


print("Connecting to SDMX services…")
CLIENT, SERVICE = connect()
DEMO_MODE = isinstance(CLIENT, DemoClient)
callout(f"Service in use : <b>{SERVICE['name']}</b><br><code>{SERVICE['url']}</code><br><i>{SERVICE['note']}</i>",
        "Connected", "warning" if DEMO_MODE else "success")
if DEMO_MODE:
    callout("The notebook runs on <b>fictional data</b> (5 dataflows): the agent's mechanics are real, "
            "the figures are not. Run it from the AfDB network to query the NSI service.",
            "Demonstration mode", "warning")

Connecting to SDMX services…
   ⚠️ AfDB · NSI Web Service : error HTTP 403
   ⚠️ .Stat · public demonstration service : error HTTP 403


   ⚠️ Pacific Data Hub · .Stat : error HTTP 403
   ⚠️ no service reachable: falling back to the demonstration snapshot


## Step 4 · Discover the **whole** catalog

**🎯 Goal:** do not settle for a single dataset. The agent must be able to reach **everything the service publishes**.

Three levels of discovery, from cheapest to most expensive:

| Level | Call | Cost | What you get |
|:--|:--|:--:|:--|
| **① Catalogue** | `GET /dataflow/{agency}/all/latest?detail=allstubs` | 1 request | every dataset and its label |
| **② Structures** | `GET /dataflow/{agency}/{flow}/latest?references=all&detail=referencepartial` | 1 request **per dataflow** | dimensions and codes actually used |
| **③ Data** | `GET /data/{agency},{flow},{version}/{key}` | 1 request per extraction | the observations |

Level ② is the costly one: on a service publishing 200 dataflows, full indexing means 200 requests. The notebook handles this with three precautions: a **disk cache** (a structure is read once), an adjustable **ceiling** (`MAX_FLOWS_INDEXED`), and **lazy indexing** — a dataflow that was not indexed will be as soon as the agent needs it.

**🧠 What indexing buys you.** It builds a directory `label → (dataflow, dimension, code)`. The agent can then answer “which dataset contains literacy?” without guessing, and above all **choose the right dataflow itself** before building its key.

In [5]:
# ── Step 4 · The catalogue and its index ────────────────────────────────────
MAX_FLOWS_INDEXED = 12        # indexing ceiling at start-up (raise it if the service is small)
INDEX_EVERYTHING = False         # True = index every dataflow, however many there are


def normalise_flow(value):
    """Accepts "DF_POP", "DEMO,DF_POP,1.0", "DEMO:DF_POP(1.0)" or a URN → returns (agency, flow)."""
    text = str(value or "").strip()
    if not text:
        return None, None
    if text.startswith("urn:"):
        text = text.split("=")[-1]
    text = text.replace("(", ",").replace(")", "").replace(":", ",")
    morceaux = [m for m in text.split(",") if m]
    if len(morceaux) == 1:
        return None, morceaux[0]
    agency, flow = morceaux[0], morceaux[1]
    return agency, flow


class SDMXCatalog:
    """Full service catalog: dataflows, cached structures and a code index."""

    def __init__(self, client, agency="all"):
        self.client, self.agency = client, agency
        self.flow = client.dataflows(agency)
        self._structures, self._failures = {}, {}
        self.index = pd.DataFrame(columns=["flow", "flow_name", "dimension", "code", "label", "label_norm"])

    # ── structures, with memory cache + the client's disk cache ────────────
    def structure(self, flow, agency=None):
        extracted_agency, flow = normalise_flow(flow)     # tolerates “AGENCY,FLOW,VERSION”
        agency = agency or extracted_agency
        if flow in self._structures:
            return self._structures[flow]
        if flow in self._failures:
            raise SDMXError(self._failures[flow])
        row = self.flow[self.flow["flow"] == flow]
        agency = agency or (row.iloc[0]["agency"] if not row.empty else self.agency)
        try:
            structure = self.client.structure(agency, flow)
        except Exception as e:
            self._failures[flow] = str(e)
            raise
        self._structures[flow] = structure
        return structure

    # ── indexation ─────────────────────────────────────────────────────────
    def index_flows(self, flows_to_index=None, verbose=True):
        targets = flows_to_index or list(self.flow["flow"])
        if not INDEX_EVERYTHING and flows_to_index is None:
            targets = targets[:MAX_FLOWS_INDEXED]
        rows, indexed = [], 0
        for flow in targets:
            if flow in self._structures or flow in self._failures:
                continue
            try:
                structure = self.structure(flow)
            except Exception:
                continue
            indexed += 1
            for dim in structure["dimensions"]:
                for c in dim["codes"]:
                    rows.append({"flow": flow, "flow_name": structure["name"], "dimension": dim["id"],
                                   "code": c["code"], "label": c["label"], "label_norm": _norm(c["label"])})
        if rows:
            self.index = pd.concat([self.index, pd.DataFrame(rows)], ignore_index=True).drop_duplicates()
        if verbose:
            print(f"   {indexed} dataflow(s) indexed · {len(self.index)} code(s) in total"
                  + (f" · {len(self._failures)} failed" if self._failures else ""))
        return self.index

    # ── search ──────────────────────────────────────────────────────────
    @staticmethod
    def _words(text):
        stop_words = {"the", "a", "an", "of", "to", "in", "on", "for", "and", "or", "is", "are", "with", "from", "by",
                 "which", "what", "how", "many", "much", "rate", "year", "since", "between"}
        return {m for m in re.findall(r"[a-z0-9]+", _norm(text)) if len(m) > 2 and m not in stop_words}

    def search_flows(self, text, k=5):
        """Ranks dataflows by relevance: the flow label plus its indexed code labels."""
        words = self._words(text)
        scores = {}
        for row in self.flow.itertuples():
            score = 2.0 * len(words & self._words(f"{row.name} {row.flow}"))
            scores[row.flow] = {"flow": row.flow, "agency": row.agency, "name": row.name,
                                  "score": score, "correspondances": ""}
        if not self.index.empty:
            for row in self.index.itertuples():
                common = words & self._words(row.label)
                if common and row.flow in scores:
                    scores[row.flow]["score"] += 1.0 * len(common)
                    if len(scores[row.flow]["correspondances"]) < 60:
                        scores[row.flow]["correspondances"] += f"{row.dimension}:{row.code} "
        table = pd.DataFrame(scores.values()).sort_values("score", ascending=False)
        return table[table["score"] > 0].head(k).reset_index(drop=True)

    def search_code(self, text, flow=None, dimension=None, k=12):
        """Searches a code in the index (across all dataflows by default)."""
        if self.index.empty:
            return self.index
        words, table = self._words(text), self.index
        if flow:
            table = table[table["flow"] == flow]
        if dimension:
            table = table[table["dimension"].str.upper() == dimension.upper()]
        reason = _norm(text)
        exact = table[table["label_norm"].str.contains(re.escape(reason), na=False) |
                      (table["code"].str.lower() == reason)]
        if not exact.empty:
            return exact.head(k)
        table = table.assign(score=table["label_norm"].map(lambda l: len(words & self._words(l))))
        return table[table["score"] > 0].sort_values("score", ascending=False).head(k)


print("Building the catalogue…")
CATALOG = SDMXCatalog(CLIENT, SERVICE.get("agency", "all"))
print(f"   {len(CATALOG.flow)} dataflow(s) published by the service")
CATALOG.index_flows()
show_table(CATALOG.flow, f"Full catalogue: {len(CATALOG.flow)} dataset(s) exposed by {SERVICE['name']}.",
        max_rows=15)

Building the catalogue…
   5 dataflow(s) published by the service
   5 dataflow(s) indexed · 55 code(s) in total


agency,flow,version,name
DEMO,DF_EDU_DEMO,1.0,Education (demonstration)
DEMO,DF_HEALTH_DEMO,1.0,Health (demonstration)
DEMO,DF_MACRO_DEMO,1.0,Macroeconomic indicators (demonstration)
DEMO,DF_POP_DEMO,1.0,Population and demography (demonstration)
DEMO,DF_TRADE_DEMO,1.0,External trade and commodities (demonstration)


In [6]:
# ── Step 4 (cont.) · What the index contains ────────────────────────────────
if not CATALOG.index.empty:
    summary = (CATALOG.index.groupby(["flow", "flow_name"])
              .agg(dimensions=("dimension", "nunique"), codes=("code", "count")).reset_index())
    show_table(summary, "Indexed dataflows: number of dimensions and codes available for each.", max_rows=15)
    examples = CATALOG.index.sample(min(8, len(CATALOG.index)), random_state=0)[["flow", "dimension", "code", "label"]]
    show_table(examples, "Extract of the “label → (dataflow, dimension, code)” directory the agent will query.")
else:
    callout("No code indexed: the service returned no structures. The agent will index on demand.",
            "Empty index", "warning")

# Default working dataflow: the first in the catalogue, or the one set by the environment variable
DEFAULT_FLOW = os.environ.get("STG17_FLUX") or CATALOG.flow.iloc[0]["flow"]
DEFAULT_AGENCY = CATALOG.flow[CATALOG.flow["flow"] == DEFAULT_FLOW].iloc[0]["agency"]
STRUCTURE = CATALOG.structure(DEFAULT_FLOW, DEFAULT_AGENCY)
callout(f"Starting dataflow: <b>{STRUCTURE['name']}</b> "
        f"(<code>{DEFAULT_AGENCY},{DEFAULT_FLOW}</code>) · key: <code>"
        f"{'.'.join(d['id'] for d in STRUCTURE['dimensions'])}</code><br>"
        "The agent can switch on its own with the <code>search_dataset</code> tool.",
        "Starting point", "success")

flow,flow_name,dimensions,codes
DF_EDU_DEMO,Education (demonstration),3,11
DF_HEALTH_DEMO,Health (demonstration),3,10
DF_MACRO_DEMO,Macroeconomic indicators (demonstration),3,12
DF_POP_DEMO,Population and demography (demonstration),3,11
DF_TRADE_DEMO,External trade and commodities (demonstration),3,11


flow,dimension,code,label
DF_TRADE_DEMO,REF_AREA,CIV,Cote d'Ivoire
DF_POP_DEMO,FREQ,A,Annual
DF_POP_DEMO,REF_AREA,KEN,Kenya
DF_MACRO_DEMO,REF_AREA,TUN,Tunisia
DF_HEALTH_DEMO,FREQ,A,Annual
DF_EDU_DEMO,REF_AREA,SEN,Senegal
DF_MACRO_DEMO,INDICATOR,GDP_CAPITA,"GDP per capita, current USD"
DF_POP_DEMO,INDICATOR,POP_GROWTH,Annual population growth


---
# 🟨 PART 3 — Equip the agent

## Step 5 · Tools wired to the **whole catalog**

**🎯 Goal:** give the agent capabilities that span everything the service publishes, not a single dataset.

Nine tools, four families:

| Tool | Family | What it does |
|:--|:--:|:--|
| `search_dataset` | 🗂️ catalog | ranks **all** dataflows by relevance to a question |
| `describe_dataflow` | 🔎 metadata | dimensions and codes of **any** dataflow |
| `search_code` | 🔎 metadata | searches the **global** index: returns dataflow + dimension + code |
| `available_values` | 🔎 metadata | codes that **actually** have data (`/availableconstraint`) |
| `fetch_data` | 📈 data | builds the key and queries **any** dataflow |
| `analyse_dataset` | 🧮 computation | latest value, change, mean, ranking |
| `join_datasets` | 🧮 computation | combines **two dataflows** on country and year |
| `plot_chart` | 📊 rendering | produces a figure from a dataset |
| `export_csv` | ✍️ **write** | saves a dataset — **subject to approval** |

**🧠 Three design choices:**

- **The dataflow is a parameter, not a constant.** Every data tool accepts `flow`; the agent can switch datasets mid-plan, and even combine two.
- **“Text → code” translation goes through the global index.** `search_code("literacy")` returns the dataflow *and* the code, so the agent never has to guess where to look.
- **Large volumes never reach the model.** Tables are kept in a store under an identifier (`dataset#1`); the model only receives a summary.

In [7]:
# ── Step 5 · The tools ──────────────────────────────────────────────────────
@dataclass
class Tool:
    name: str
    description: str
    parameters: dict
    function: Callable
    writes: bool = False

    def run_tool(self, **arguments):
        unknown = set(arguments) - set(self.parameters)
        if unknown:
            raise TypeError(f"unknown parameter(s): {sorted(unknown)}; expected: {sorted(self.parameters)}")
        return str(self.function(**arguments))


class DatasetStore:
    """Keeps the fetched tables; the model only handles their identifiers."""

    def __init__(self):
        self.datasets, self.figures = {}, {}

    def add(self, table, meta):
        identifier = f"dataset#{len(self.datasets) + 1}"
        self.datasets[identifier] = {"table": table, "meta": meta}
        return identifier

    def table(self, identifier):
        if identifier not in self.datasets:
            raise KeyError(f"dataset inconnu : {identifier} (disponibles : {', '.join(self.datasets) or 'none'})")
        return self.datasets[identifier]["table"]

    def clear(self):
        self.datasets.clear(); self.figures.clear()


STORE = DatasetStore()
COLOURS = [GREEN, NAVY, GOLD, RED, "#0E7C86", "#C4621D", "#7A5BA6"]


def build_tools(catalog, current_flow, store=STORE, folder=OUTPUTS):
    """Builds the tools FROM the whole catalogue; `current_flow` is only a default."""

    def _structure(flow=None):
        _, identifier = normalise_flow(flow) if flow else (None, None)
        return catalog.structure(identifier or current_flow)

    def _reference(flow):
        st = _structure(flow)
        return st["agency"], st["flow"], st["version"], st

    # ── 🗂️ catalog ───────────────────────────────────────────────────────
    def search_dataset(question):
        results = catalog.search_flows(question, k=6)
        if results.empty:
            preview = catalog.flow.head(10)
            return ("no match; available dataflows:\n"
                    + "\n".join(f"{r.flow} — {r.name}" for r in preview.itertuples()))
        rows = []
        for r in results.itertuples():
            indexe = r.flow in catalog._structures
            rows.append(f"{r.flow} — {r.name} (score {r.score:g}"
                          + (f", codes : {r.correspondances.strip()}" if r.correspondances else "")
                          + (")" if indexe else ", not indexed)"))
        return "\n".join(rows)

    # ── 🔎 metadata ────────────────────────────────────────────────────────
    def describe_dataflow(flow=""):
        st = _structure(flow or None)
        rows = [f"Dataflow {st['agency']},{st['flow']},{st['version']} — {st['name']}",
                  f"Key order: {'.'.join(d['id'] for d in st['dimensions'])}",
                  f"Time dimension: {st['dimension_temps']}"]
        for d in st["dimensions"]:
            examples = ", ".join(f"{c['code']}={c['label']}" for c in d["codes"][:8])
            rows.append(f"- {d['id']} ({len(d['codes'])} codes) : {examples}{' …' if len(d['codes']) > 8 else ''}")
        return "\n".join(rows)

    def search_code(text, dimension="", flow=""):
        flow = normalise_flow(flow)[1] or ""
        if flow:                                   # make sure this dataflow is indexed
            try:
                catalog.index_flows([flow], verbose=False)
            except Exception:
                pass
        found = catalog.search_code(text, flow=flow or None, dimension=dimension or None)
        if found.empty and not flow:
            catalog.index_flows(list(catalog.flow["flow"])[:MAX_FLOWS_INDEXED * 2], verbose=False)
            found = catalog.search_code(text, dimension=dimension or None)
        if found.empty:
            return f"no code matches “{text}” in the indexed dataflows"
        return "\n".join(f"{r.flow} · {r.dimension} · {r.code} = {r.label}" for r in found.itertuples())

    def available_values(dimension, flow="", key=""):
        """Codes that actually hold data, via /availableconstraint (falls back to the codelist)."""
        agency, identifier, version, st = _reference(flow or None)
        try:
            payload = catalog.client.get(f"availableconstraint/{agency},{identifier},{version}/"
                                          f"{key or 'all'}/all/{dimension}", {"mode": "available"})
            root = catalog.client._root(payload)
            values = []
            for contrainte in root.get("contentConstraints", root.get("dataConstraints", [])):
                for region in contrainte.get("cubeRegions", []):
                    for membre in region.get("keyValues", []):
                        if membre.get("id", "").upper() == dimension.upper():
                            values += [v if isinstance(v, str) else v.get("value", "") for v in membre.get("values", [])]
            if values:
                labels = {c["code"]: c["label"] for d in st["dimensions"] if d["id"] == dimension for c in d["codes"]}
                return (f"{len(values)} value(s) with data for {dimension}: "
                        + ", ".join(f"{v}={labels.get(v, '')}" for v in values[:25]))
        except Exception as e:
            pass
        codes = [c for d in st["dimensions"] if d["id"].upper() == dimension.upper() for c in d["codes"]]
        if not codes:
            return f"unknown dimension: {dimension} (dimensions: {', '.join(d['id'] for d in st['dimensions'])})"
        return (f"availability not provided by the service; declared codes for {dimension} : "
                + ", ".join(f"{c['code']}={c['label']}" for c in codes[:25]))

    # ── 📈 data ────────────────────────────────────────────────────────────
    def fetch_data(key="all", flow="", start=None, end=None, last_n=None):
        agency, identifier, version, st = _reference(flow or None)
        table = catalog.client.data(agency, identifier, key=key, version=version,
                                         start=start, end=end, last_n=last_n)
        if table.empty:
            return (f"NO DATA for “{key}” in {identifier}: check the dimension order "
                    f"({'.'.join(d['id'] for d in st['dimensions'])}) or the period.")
        if len(table) > MAX_OBSERVATIONS:
            return (f"REFUSED: {len(table)} observations exceed the {MAX_OBSERVATIONS} limit. "
                    "Narrow the key or the period.")
        dataset_id = store.add(table, {"key": key, "start": start, "end": end, "flow": identifier,
                                                  "structure": st, "service": SERVICE["name"]})
        columns = [c for c in table.columns if not c.endswith("_LIBELLE")]
        preview = "; ".join(" ".join(str(r[c]) for c in columns) for _, r in table.head(2).iterrows())
        temps = st["dimension_temps"]
        periods = sorted(table[temps].unique()) if temps in table else []
        return (f"{dataset_id} : {len(table)} observations from {identifier} for “{key or 'all'}” "
                f"(period {periods[0] if periods else '?'} → {periods[-1] if periods else '?'}) · "
                f"columns {', '.join(columns)} · examples : {preview}")

    # ── 🧮 calcul ──────────────────────────────────────────────────────────
    def _time_dim(table, meta):
        st = meta.get("structure") or _structure()
        return st["dimension_temps"] if st["dimension_temps"] in table.columns else \
            next((c for c in table.columns if c.startswith("TIME")), None)

    def analyse_dataset(dataset, operation="latest_value", group=None):
        entry = store.datasets.get(dataset)
        if entry is None:
            return f"ERROR: unknown dataset {dataset}"
        table = entry["table"].copy()
        temps = _time_dim(table, entry["meta"])
        dims = [c for c in table.columns if not c.endswith("_LIBELLE") and c not in (temps, "OBS_VALUE")]
        varying = [c for c in dims if table[c].nunique() > 1]
        keys = [c for c in (group.split(",") if group else varying) if c in table.columns] or \
               [c for c in dims if c != "FREQ"] or dims[:1]
        table = table.sort_values(temps)
        if operation == "latest_value":
            res = table.groupby(keys, as_index=False).tail(1)[keys + [temps, "OBS_VALUE"]]
        elif operation == "change":
            def change(g):
                g = g.dropna(subset=["OBS_VALUE"])
                if len(g) < 2:
                    return pd.Series({"depuis": None, "jusqu_a": None, "change": None, "change_pct": None})
                depart, arrivee = g.iloc[0]["OBS_VALUE"], g.iloc[-1]["OBS_VALUE"]
                return pd.Series({"depuis": g.iloc[0][temps], "jusqu_a": g.iloc[-1][temps],
                                  "change": round(arrivee - depart, 3),
                                  "change_pct": (round((arrivee / depart - 1) * 100, 1) if depart else None)})
            res = table.groupby(keys).apply(change, include_groups=False).reset_index()
        elif operation == "mean":
            res = table.groupby(keys, as_index=False)["OBS_VALUE"].mean().round(3)
        elif operation == "ranking":
            derniers = table.groupby(keys, as_index=False).tail(1)
            res = derniers.sort_values("OBS_VALUE", ascending=False)[keys + [temps, "OBS_VALUE"]]
        else:
            return f"ERROR: unknown operation “{operation}” (latest_value, change, mean, ranking)"
        nouveau = store.add(res, {"origine": dataset, "operation": operation, "structure": entry["meta"].get("structure")})
        return f"{nouveau} : {operation} sur {dataset}\n" + res.head(12).to_string(index=False)

    def join_datasets(jeu_a, jeu_b, sur="REF_AREA,TIME_PERIOD"):
        """Combines two extractions — including from different dataflows."""
        columns = [c.strip() for c in sur.split(",") if c.strip()]
        a, b = store.table(jeu_a).copy(), store.table(jeu_b).copy()
        missing = [c for c in columns if c not in a.columns or c not in b.columns]
        if missing:
            return (f"ERROR: columns missing from one dataset: {missing}. "
                    f"{jeu_a} : {list(a.columns)} · {jeu_b} : {list(b.columns)}")
        def prepare(table, suffix):
            indicator = next((c for c in ("INDICATOR", "SERIES", "MEASURE") if c in table.columns), None)
            name = (table[indicator].iloc[0] if indicator is not None and not table.empty else suffix)
            return table[columns + ["OBS_VALUE"]].rename(columns={"OBS_VALUE": str(name)})
        merged = prepare(a, "A").merge(prepare(b, "B"), on=columns, how="inner")
        if merged.empty:
            return "NO match between the two datasets on " + ", ".join(columns)
        identifier = store.add(merged, {"origine": f"{jeu_a}+{jeu_b}", "jointure": columns})
        return f"{identifier}: {len(merged)} row(s) combined on {', '.join(columns)}\n" + merged.head(10).to_string(index=False)

    # ── 📊 rendu ───────────────────────────────────────────────────────────
    def plot_chart(dataset, title="", chart_type="row"):
        entry = store.datasets.get(dataset)
        if entry is None:
            return f"ERROR: unknown dataset {dataset}"
        table = entry["table"].copy()
        temps = _time_dim(table, entry["meta"])
        if temps is None or "OBS_VALUE" not in table.columns:
            colonnes_num = [c for c in table.columns if pd.api.types.is_numeric_dtype(table[c])]
            if temps is None or len(colonnes_num) < 1:
                return "ERROR: this dataset has no plottable time series"
        series = [c for c in table.columns
                  if not c.endswith("_LIBELLE") and c not in (temps, "OBS_VALUE") and table[c].nunique() > 1]
        fig, ax = plt.subplots(figsize=(8.2, 3.6))
        if "OBS_VALUE" in table.columns and series:
            for i, (name, group) in enumerate(table.groupby(series[0])):
                label = group.get(series[0] + "_LIBELLE", pd.Series([name])).iloc[0]
                group = group.sort_values(temps)
                colour = COLOURS[i % len(COLOURS)]
                if chart_type == "barres":
                    ax.bar(group[temps].astype(str), group["OBS_VALUE"], color=colour, label=str(label))
                else:
                    ax.plot(group[temps].astype(str), group["OBS_VALUE"], marker="o", ms=4, color=colour, label=str(label))
                    if not group.empty:
                        ax.annotate(f"{group['OBS_VALUE'].iloc[-1]:g}", (len(group) - 1, group["OBS_VALUE"].iloc[-1]),
                                    fontsize=8.5, color=colour, fontweight="bold", xytext=(4, 2), textcoords="offset points")
            ax.legend(frameon=False, fontsize=8.5, ncol=2)
        elif "OBS_VALUE" in table.columns:
            ax.plot(table[temps].astype(str), table["OBS_VALUE"], marker="o", ms=4, color=GREEN)
        else:                                        # dataset from a join: one line per numeric column
            for i, column in enumerate([c for c in table.columns if pd.api.types.is_numeric_dtype(table[c])]):
                ax.plot(table[temps].astype(str), table[column], marker="o", ms=4,
                        color=COLOURS[i % len(COLOURS)], label=column)
            ax.legend(frameon=False, fontsize=8.5)
        ax.set_title(title or entry["meta"].get("key", dataset))
        ax.grid(axis="y", color="#E1E7E4")
        ax.set_xlabel(temps)
        plt.xticks(rotation=45, ha="right", fontsize=8)
        plt.tight_layout()
        tampon = io.BytesIO()
        fig.savefig(tampon, format="png", dpi=120)
        plt.close(fig)
        store.figures[dataset] = base64.b64encode(tampon.getvalue()).decode()
        return f"figure produced for {dataset} ({len(table)} points)"

    # ── ✍️ write ───────────────────────────────────────────────────────────
    def export_csv(dataset, file_name):
        name = Path(str(file_name)).name
        if not re.fullmatch(r"[\w\-]{1,60}\.csv", name):
            return f"ERROR: file name rejected ({file_name!r})"
        Path(folder).mkdir(parents=True, exist_ok=True)
        store.table(dataset).to_csv(Path(folder) / name, index=False)
        return f"export done: {name} ({len(store.table(dataset))} rows)"

    st_defaut = _structure()
    order = ".".join(d["id"] for d in st_defaut["dimensions"])
    return [
        Tool("search_dataset", "Ranks ALL the service's dataflows by relevance to a question.",
              {"question": "the question or keywords"}, search_dataset),
        Tool("describe_dataflow", "Dimensions and codes of a dataflow (the current one if empty).",
              {"flow": "dataflow identifier, or empty"}, describe_dataflow),
        Tool("search_code", "Searches a label in the global index: returns dataflow, dimension and code.",
              {"text": "e.g. literacy", "dimension": "e.g. REF_AREA (optional)",
               "flow": "restrict to one dataflow (optional)"}, search_code),
        Tool("available_values", "Codes that actually have data for a dimension.",
              {"dimension": "ex. REF_AREA", "flow": "dataflow (optional)", "key": "partial key (optional)"},
              available_values),
        Tool("fetch_data", f"Fetches observations. Key order of the current dataflow: {order}.",
              {"key": "ex. A.CIV.GDP_GROWTH", "flow": "dataflow (optional)", "start": "start year",
               "end": "end year", "last_n": "number of most recent observations"}, fetch_data),
        Tool("analyse_dataset", "Computes on a dataset already fetched.",
              {"dataset": "ex. dataset#1", "operation": "latest_value | change | mean | ranking",
               "group": "dimensions separated by commas (optional)"}, analyse_dataset),
        Tool("join_datasets", "Combines two datasets, including from different dataflows.",
              {"jeu_a": "ex. dataset#1", "jeu_b": "ex. dataset#2", "sur": "columns de jointure"}, join_datasets),
        Tool("plot_chart", "Produces a chart from a dataset.",
              {"dataset": "ex. dataset#1", "title": "title", "chart_type": "row | barres"}, plot_chart),
        Tool("export_csv", "Saves a dataset as CSV (write).",
              {"dataset": "ex. dataset#1", "file_name": "e.g. extract.csv"}, export_csv, writes=True),
    ]


TOOLS = build_tools(CATALOG, DEFAULT_FLOW)
TOOLS_BY_NAME = {o.name: o for o in TOOLS}
show_table(pd.DataFrame([{"tool": o.name, "type": "✍️ write" if o.writes else "👁️ read",
                       "parameters": ", ".join(o.parameters), "description": o.description} for o in TOOLS]),
        "These tools span the whole catalogue: the dataflow is a parameter, not a constant.")

tool,type,parameters,description
search_dataset,👁️ read,question,Ranks ALL the service's dataflows by relevance to a question.
describe_dataflow,👁️ read,flow,Dimensions and codes of a dataflow (the current one if empty).
search_code,👁️ read,"text, dimension, flow","Searches a label in the global index: returns dataflow, dimension and code."
available_values,👁️ read,"dimension, flow, key",Codes that actually have data for a dimension.
fetch_data,👁️ read,"key, flow, start, end, last_n",Fetches observations. Key order of the current dataflow: FREQ.REF_AREA.INDICATOR.
analyse_dataset,👁️ read,"dataset, operation, group",Computes on a dataset already fetched.
join_datasets,👁️ read,"jeu_a, jeu_b, sur","Combines two datasets, including from different dataflows."
plot_chart,👁️ read,"dataset, title, chart_type",Produces a chart from a dataset.
export_csv,✍️ write,"dataset, file_name",Saves a dataset as CSV (write).


In [8]:
# ── Step 5 (cont.) · Trying the tools by hand ───────────────────────────────
for name, args in [("search_dataset", {"question": "urban population"}),
                  ("search_code", {"text": "literacy"}),
                  ("describe_dataflow", {})]:
    print(f"▶ {name}({args})\n{TOOLS_BY_NAME[name].run_tool(**args)[:500]}\n")

▶ search_dataset({'question': 'urban population'})
DF_POP_DEMO — Population and demography (demonstration) (score 6, codes : INDICATOR:POP_TOT INDICATOR:URBAN_PCT INDICATOR:POP_GROWTH)

▶ search_code({'text': 'literacy'})
DF_EDU_DEMO · INDICATOR · LITERACY = Adult literacy rate

▶ describe_dataflow({})
Dataflow DEMO,DF_EDU_DEMO,latest — Education (demonstration)
Key order: FREQ.REF_AREA.INDICATOR
Time dimension: TIME_PERIOD
- FREQ (1 codes) : A=Annual
- REF_AREA (7 codes) : CIV=Cote d'Ivoire, SEN=Senegal, GHA=Ghana, NGA=Nigeria, TUN=Tunisia, MAR=Morocco, KEN=Kenya
- INDICATOR (3 codes) : LITERACY=Adult literacy rate, COMPLETION_PRIM=Primary completion rate, PUPIL_TEACHER=Pupil / teacher ratio



## Step 6 · The protocol and **planning**

**🎯 Goal:** make the agent **think before it calls**, instead of groping request after request.

We use the **Plan-and-Execute** pattern, in three stages:

1. **PLAN** — one model call produces a plan, as a list of steps with the tool and the goal of each:

```json
{"plan": [
  {"goal": "find the country code", "tool": "search_code", "args": {"dimension": "REF_AREA", "text": "Cote d'Ivoire"}},
  {"goal": "fetch the series", "tool": "fetch_data", "args": {"key": "A.CIV.GDP_GROWTH", "start": "2015"}},
  {"goal": "plot", "tool": "plot_chart", "args": {"dataset": "dataset#1"}}
]}
```

2. **EXECUTE** — each step goes through the policy, then the tool. A step can **reuse an earlier result** through the `{{dataset}}` and `{{code}}` placeholders.

3. **REPLAN** — if a step fails (unknown code, invalid key, no data), the error is sent back to the model, which proposes a **fix**: a new step, or a revised plan. Beyond `MAX_CORRECTIONS`, the agent stops and says so.

**Why plan rather than react?** Three concrete reasons: the **cost** is predictable (you know before starting how many calls will be made), the plan is **readable by a human** before execution, and it can be **submitted for approval** in sensitive cases.

In [9]:
# ── Step 6 · Protocol: parsing the model's replies ──────────────────────────
MAX_CORRECTIONS = 3

class ProtocolError(Exception):
    """The model reply cannot be used."""


def extract_json(text):
    """First balanced JSON object, even wrapped in text or ``` fences."""
    start = text.find("{")
    while start != -1:
        profondeur, dans_chaine, echappe = 0, False, False
        for i in range(start, len(text)):
            c = text[i]
            if dans_chaine:
                echappe = (c == "\\") and not echappe
                if c == '"' and not echappe:
                    dans_chaine = False
                continue
            if c == '"':
                dans_chaine = True
            elif c == "{":
                profondeur += 1
            elif c == "}":
                profondeur -= 1
                if profondeur == 0:
                    try:
                        return json.loads(text[start:i + 1])
                    except json.JSONDecodeError:
                        break
        start = text.find("{", start + 1)
    raise ProtocolError("no usable JSON object")


def parse_plan(text):
    obj = extract_json(text)
    if "plan" not in obj or not isinstance(obj["plan"], list) or not obj["plan"]:
        raise ProtocolError("reply without a usable “plan” key")
    plan = []
    for i, step in enumerate(obj["plan"][:8], start=1):
        if not isinstance(step, dict) or "tool" not in step:
            raise ProtocolError(f"step {i} without “tool”")
        args = step.get("args", {})
        if not isinstance(args, dict):
            raise ProtocolError(f"step {i}: “args” must be an object")
        plan.append({"number": i, "goal": str(step.get("goal", "")), "tool": str(step["tool"]), "args": args})
    return plan


def parse_correction(text):
    """After a failure: one step, a full replacement plan, or a reasoned give-up."""
    obj = extract_json(text)
    if "give_up" in obj:
        return None, str(obj["give_up"])
    if "plan" in obj:
        return parse_plan(text), ""
    if "tool" in obj:
        return [{"number": 0, "goal": str(obj.get("goal", "correction")), "tool": str(obj["tool"]),
                 "args": obj.get("args", {})}], ""
    raise ProtocolError("fix without “tool”, “plan” or “give_up”")


def describe_tools(tools):
    rows = []
    for o in tools:
        params = ", ".join(f"{p}: {d}" for p, d in o.parameters.items()) or "none"
        rows.append(f"- {o.name}({params}) — {o.description}{' [WRITE: subject to approval]' if o.writes else ''}")
    return "\n".join(rows)


PLAN_INSTRUCTIONS = """You are a statistical analysis agent connected to a SDMX service.
Service: <<SERVICE>> · catalogue: <<CATALOG>> published datasets
Starting dataflow (you may switch it with the “flow” parameter): <<FLOWS>>
Key order: <<ORDER>>
Dimensions et examples de codes :
<<DIMENSIONS>>

Available tools:
<<TOOLS>>

Reply with ONE JSON object and no other text:
{"plan": [{"goal": "...", "tool": "<name>", "args": {...}}, ...]}

Rules:
1. At most 6 steps.
2. NEVER invent a code: use search_code, or reuse a code already returned.
3. To chain steps, use {{dataset}}, replaced by the identifier of the last dataset fetched.
4. Always restrict the period (start/end) or the number of observations.
5. Finish with plot_chart when the question is about a trend or a comparison.
6. If the question does not belong to the starting dataflow, begin with search_dataset, then pass
   the right identifier in “flow” to fetch_data.
   IMPORTANT : “flow” attend l’identifier SEUL (ex. "DF_POP_DEMO"), jamais "AGENCE,FLOWS,VERSION".
   “{{dataset}}” is only valid AFTER a successful fetch_data: do not use it in the first step.
7. To combine two indicators from different dataflows: two fetch_data calls, then join_datasets."""

SUMMARY_INSTRUCTIONS = """You write the final answer from the TOOL RESULTS below, in English.
Rules:
1. Use ONLY the figures present in these results; never recompute anything mentally.
2. Cite the source at the end, in brackets: [<<SOURCE>>].
3. Three sentences at most, then optionally a short list.
4. If the results do not allow an answer, say so plainly."""


def plan_instructions(tools, structure, service):
    dimensions = "\n".join(
        f"  {d['id']} : " + ", ".join(f"{c['code']}={c['label']}" for c in d["codes"][:8]) +
        (" …" if len(d["codes"]) > 8 else "") for d in structure["dimensions"])
    return (PLAN_INSTRUCTIONS
            .replace("<<SERVICE>>", service["name"])
            .replace("<<CATALOG>>", str(len(CATALOG.flow)))
            .replace("<<FLOWS>>", f"{structure['agency']},{structure['flow']},{structure['version']} — {structure['name']}")
            .replace("<<ORDRE>>", ".".join(d["id"] for d in structure["dimensions"]))
            .replace("<<DIMENSIONS>>", dimensions)
            .replace("<<TOOLS>>", describe_tools(tools)))


print(plan_instructions(TOOLS, STRUCTURE, SERVICE)[:1500], "\n…")

You are a statistical analysis agent connected to a SDMX service.
Service: Demonstration snapshot · catalogue: 5 published datasets
Starting dataflow (you may switch it with the “flow” parameter): DEMO,DF_EDU_DEMO,latest — Education (demonstration)
Key order: <<ORDER>>
Dimensions et examples de codes :
  FREQ : A=Annual
  REF_AREA : CIV=Cote d'Ivoire, SEN=Senegal, GHA=Ghana, NGA=Nigeria, TUN=Tunisia, MAR=Morocco, KEN=Kenya
  INDICATOR : LITERACY=Adult literacy rate, COMPLETION_PRIM=Primary completion rate, PUPIL_TEACHER=Pupil / teacher ratio

Available tools:
- search_dataset(question: the question or keywords) — Ranks ALL the service's dataflows by relevance to a question.
- describe_dataflow(flow: dataflow identifier, or empty) — Dimensions and codes of a dataflow (the current one if empty).
- search_code(text: e.g. literacy, dimension: e.g. REF_AREA (optional), flow: restrict to one dataflow (optional)) — Searches a label in the global index: returns dataflow, dimension and code.
- 

## Step 7 · Policy, budget and log

**🎯 Goal:** put guard rails around an agent that now talks to an **external service**.

Three new risks appear compared with a documentary agent:

| Risk | Guard rail |
|:--|:--|
| A huge request overloading the service | `MAX_OBSERVATIONS` limit, refusal of the `all` key without a period |
| A loop of network calls | a budget in steps **and** a budget in HTTP requests |
| Writing files | `export_csv` goes through the **approval queue** |

And the audit log records, on top of the agent's steps, **the exact URL of every SDMX request**: that is what makes an answer reproducible by a third party.

In [10]:
# ── Step 7 · Policy, approval queue, audit log ──────────────────────────────
def service_policy(tool, args):
    """Refuses what endangers the service or the data; queues writes."""
    if tool.name == "fetch_data":
        key = str(args.get("key", "") or "all")
        sans_periode = not args.get("start") and not args.get("end") and not args.get("last_n")
        if key in ("", "all") and sans_periode:
            return False, "request too broad: give a key or a period"
    if tool.writes:
        return None, "write subject to human approval"      # None = mise en waiting
    return True, "read allowed"


@dataclass
class ApprovalRequest:
    number: int
    tool: str
    args: dict
    task: str
    created_at: str
    status: str = "en waiting"
    decided_by: str = ""
    result: str = ""


class ApprovalQueue:
    def __init__(self, outils_par_nom):
        self.tools, self.requests, self.decisions = outils_par_nom, [], []
        self.current_task = ""

    def policy(self, tool, args):
        allowed, reason = service_policy(tool, args)
        if allowed is None:
            request = ApprovalRequest(len(self.requests) + 1, tool.name, dict(args), self.current_task,
                                         datetime.now(timezone.utc).isoformat(timespec="seconds"))
            self.requests.append(request)
            return False, f"queued for approval (request #{request.number}): nothing is written without validation"
        return allowed, reason

    def pending(self):
        return [d for d in self.requests if d.status == "en waiting"]

    def decide(self, number, approuver, decided_by="analyste"):
        request = next(d for d in self.requests if d.number == number)
        if request.status != "en waiting":
            return request
        if approuver:
            request.result = self.tools[request.tool].run_tool(**request.args)
            request.status = "approved"
        else:
            request.status = "rejected"
        request.decided_by = decided_by
        previous = self.decisions[-1]["hash"] if self.decisions else "0" * 64
        entry = {"number": number, "tool": request.tool, "args": request.args, "status": request.status,
                  "decided_by": decided_by, "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                  "previous_hash": previous}
        entry["hash"] = hashlib.sha256(json.dumps(entry, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
        self.decisions.append(entry)
        return request


QUEUE = ApprovalQueue(TOOLS_BY_NAME)


def _fingerprint(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def save_log(execution, path, meta):
    previous, entries = "0" * 64, []
    for e in execution.steps:
        entry = {"step": e.number, "goal": e.goal, "tool": e.tool, "args": e.args, "allowed": e.allowed,
                  "reason": e.reason, "result": (e.result or "")[:2000], "error": e.error,
                  "previous_hash": previous}
        entry["hash"] = previous = _fingerprint(entry)
        entries.append(entry)
    document = {"meta": {**meta, "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                         "question": execution.question, "answer": execution.answer, "stop": execution.stop,
                         "service": SERVICE["name"], "http_requests": execution.http_requests},
                "steps": entries, "seal": previous}
    Path(path).write_text(json.dumps(document, ensure_ascii=False, indent=2), encoding="utf-8")
    return path


def verify_log(path):
    document = json.loads(Path(path).read_text(encoding="utf-8"))
    previous = "0" * 64
    for entry in document["steps"]:
        copy = {k: v for k, v in entry.items() if k != "hash"}
        if copy["previous_hash"] != previous or _fingerprint(copy) != entry["hash"]:
            return False, f"chain broken at step {entry['step']}"
        previous = entry["hash"]
    return (previous == document["seal"]), ("intact" if previous == document["seal"] else "seal invalide")


cases = [("fetch_data", {"key": "all"}), ("fetch_data", {"key": "all", "start": "2015"}),
       ("export_csv", {"dataset": "dataset#1", "file_name": "extraction.csv"})]
show_table(pd.DataFrame([{"tool": n, "arguments": json.dumps(a, ensure_ascii=False),
                       "decision": "{} {}".format(*(lambda r: ("✅" if r[0] else "🕒" if "waiting" in r[1] else "⛔", r[1]))(
                           QUEUE.policy(TOOLS_BY_NAME[n], a)))} for n, a in cases]),
        "The policy protects the service (over-broad requests) and the data (writes).")
QUEUE = ApprovalQueue(TOOLS_BY_NAME)     # reset after the demonstration

tool,arguments,decision
fetch_data,"{""key"": ""all""}",⛔ request too broad: give a key or a period
fetch_data,"{""key"": ""all"", ""start"": ""2015""}",✅ read allowed
export_csv,"{""dataset"": ""dataset#1"", ""file_name"": ""extraction.csv""}",⛔ queued for approval (request #1): nothing is written without validation


## Step 7b · The Plan-and-Execute loop

**🎯 Goal:** assemble the agent's engine.

Read the `run` method: it fits on one page and holds the whole logic — plan, substitute placeholders, consult the policy, execute, fix on failure, then summarise.

In [11]:
# ── Step 7b · The planning agent ────────────────────────────────────────────
@dataclass
class ExecStep:
    number: int
    goal: str = ""
    tool: str = ""
    args: dict = field(default_factory=dict)
    allowed: Optional[bool] = None
    reason: str = ""
    result: Optional[str] = None
    error: Optional[str] = None
    duration_ms: float = 0.0


@dataclass
class Run:
    question: str
    plan: list = field(default_factory=list)
    steps: list = field(default_factory=list)
    answer: str = ""
    stop: str = ""
    corrections: int = 0
    model_calls: int = 0
    http_requests: int = 0
    raw_plan: str = ""
    datasets: list = field(default_factory=list)

    def log(self):
        return pd.DataFrame([{"step": e.number, "goal": e.goal, "tool": e.tool,
                              "arguments": json.dumps(e.args, ensure_ascii=False)[:70],
                              "decision": "" if e.allowed is None else ("✅" if e.allowed else
                                                                         ("🕒" if "waiting" in e.reason else "⛔")),
                              "result / error": (e.error or (e.result or "")).replace("\n", " ")[:90],
                              "ms": round(e.duration_ms)} for e in self.steps])


class SDMXAgent:
    def __init__(self, tools, brain, structure, service, file=None, max_steps=8, max_requests=12):
        self.tools = {o.name: o for o in tools}
        self.brain, self.structure, self.service = brain, structure, service
        self.file = file or QUEUE
        self.max_steps, self.max_requests = max_steps, max_requests
        self.instructions = plan_instructions(tools, structure, service)

    def _substitute(self, args, last_dataset, last_code):
        dataset = last_dataset or (sorted(STORE.datasets, key=lambda j: int(j.split("#")[1]))[-1] if STORE.datasets else "")
        resolus = {}
        for key, value in args.items():
            if isinstance(value, str):
                value = value.replace("{{dataset}}", dataset).replace("{{code}}", last_code or "")
            if key == "flow" and isinstance(value, str) and value:
                value = normalise_flow(value)[1] or value      # “DEMO,DF_X,latest” → “DF_X”
            resolus[key] = value
        return resolus

    def run_tool(self, question, verbose=False):
        run = Run(question=question)
        self.file.current_task = question
        requests_start = CLIENT.requests_count
        say = print if verbose else (lambda *a, **k: None)

        # ① PLAN (two attempts: “reasoning” models sometimes miss the format)
        exchanges, raw, last_error = [{"role": "user", "content": question}], "", None
        for attempt in range(2):
            try:
                raw = self.brain(self.instructions, exchanges, json_strict=True)
                run.model_calls += 1
                run.plan = parse_plan(raw)
                last_error = None
                break
            except Exception as e:
                last_error = e
                run.raw_plan = raw
                exchanges = [{"role": "user", "content": question},
                            {"role": "assistant", "content": raw[:500] or "(empty reply)"},
                            {"role": "user", "content": "Your reply was unusable. Reply ONLY with "
                                                        "a JSON object of the form "
                                                        '{"plan":[{"goal":"...","tool":"...","args":{...}}]} '
                                                        "— no text before or after."}]
        if last_error is not None:
            run.stop = "planning failure"
            run.answer = (f"I could not build a plan: {last_error}.\n"
                           + (f"Raw model reply: “{raw[:200]}”\n" if raw else "")
                           + "Try a more precise question, or switch to another brain "
                             "(par exemple `MODEL = \"openai/gpt-oss-120b\"`).")
            return run
        say("🗺️ PLAN :", *[f"  {e['number']}. {e['tool']} — {e['goal']}" for e in run.plan], sep="\n")

        # ② EXECUTE
        last_dataset, last_code, i = None, None, 0
        step_queue = list(run.plan)
        while step_queue and len(run.steps) < self.max_steps:
            plan_step = step_queue.pop(0)
            i += 1
            t0 = time.perf_counter()
            step = ExecStep(number=i, goal=plan_step["goal"], tool=plan_step["tool"],
                              args=self._substitute(plan_step["args"], last_dataset, last_code))
            run.steps.append(step)
            tool = self.tools.get(step.tool)
            if tool is None:
                step.error = f"tool inconnu : {step.tool}"
            else:
                step.allowed, step.reason = self.file.policy(tool, step.args)
                if step.allowed:
                    if CLIENT.requests_count - requests_start >= self.max_requests:
                        step.error = "HTTP request budget exhausted"
                    else:
                        try:
                            step.result = tool.run_tool(**step.args)
                        except (TypeError, KeyError, SDMXError) as e:
                            step.error = f"{type(e).__name__} : {e}"
            step.duration_ms = (time.perf_counter() - t0) * 1000
            text = step.result or ""
            trouve_jeu = re.match(r"(dataset#\d+)", text)
            if trouve_jeu:
                last_dataset = trouve_jeu.group(1)
                run.datasets.append(last_dataset)
            if step.tool == "search_code" and text and "=" in text:
                last_code = text.split("\n")[0].split("=")[0].strip()
            if step.tool == "search_dataset" and text and " — " in text:
                chosen_flow = text.split(" — ")[0].split("\n")[-1].strip()
                if chosen_flow:
                    self.tools = {o.name: o for o in build_tools(CATALOG, chosen_flow)}
                    say(f"     → dataflow de travail : {chosen_flow}")
            say(f"  {i}. {'✅' if step.allowed else '⛔'} {step.tool} → {(step.error or text)[:80]}")

            # ③ REPLAN on failure
            echec = step.error or text.startswith(("ERREUR", "AUCUNE", "REFUS", "none"))
            if echec and run.corrections < MAX_CORRECTIONS:
                run.corrections += 1
                message = (f"Step “{step.tool}” failed: {step.error or text}. "
                           "Propose ONE replacement step in the form "
                           '{"goal":"...","tool":"...","args":{...}} ou {"give_up":"raison"}.')
                try:
                    raw = self.brain(self.instructions, [{"role": "user", "content": question},
                                                        {"role": "assistant", "content": json.dumps({"plan": run.plan}, ensure_ascii=False)},
                                                        {"role": "user", "content": message}], json_strict=True)
                    run.model_calls += 1
                    corrections, give_up = parse_correction(raw)
                    if corrections:
                        for fixed_step in reversed(corrections):
                            step_queue.insert(0, fixed_step)
                        # if the failed step was fetching data, replay it AFTER the fix
                        if step.tool == "fetch_data" and not any(
                                c["tool"] == "fetch_data" for c in corrections):
                            retry_args = dict(plan_step["args"])
                            retry_args.pop("flow", None)          # laisse l’agent repartir du dataflow retenu
                            step_queue.insert(len(corrections),
                                               {"number": 0, "goal": "retrying the fetch",
                                                "tool": "fetch_data", "args": retry_args})
                        say(f"     ↻ correction : {', '.join(c['tool'] for c in corrections)}")
                    else:
                        run.stop = f"give_up : {give_up}"
                        break
                except Exception as e:
                    say(f"     ↻ correction impossible ({e})")

        run.stop = run.stop or ("step budget reached" if len(run.steps) >= self.max_steps else "plan completed")

        # ④ SUMMARISE
        observations = "\n".join(f"[{e.tool}] {(e.result or e.error or '')[:600]}" for e in run.steps)
        source = f"{self.structure['agency']},{self.structure['flow']} via {self.service['name']}"
        try:
            run.answer = self.brain(SUMMARY_INSTRUCTIONS.replace("<<SOURCE>>", source),
                                       [{"role": "user", "content": f"QUESTION: {question}\n\nRESULTS:\n{observations}"}])
            run.model_calls += 1
        except Exception as e:
            run.answer = f"Results obtained, but the summary failed ({e})."
        run.http_requests = CLIENT.requests_count - requests_start
        return run

---
# 🟪 PART 4 — The brains

## Step 8 · Local planner or free model

**🎯 Goal:** show that **planning** can come from two sources, with nothing else changing.

| Brain | Key | How it plans |
|:--|:--:|:--|
| 🧪 **Local planner** | none | it picks the **dataflow** from the whole catalog, spots the **codes** in it, then applies a template plan |
| ⚡ **Groq** · `openai/gpt-oss-20b` | `GROQ_API_KEY` | the model writes the plan itself |
| 🔷 **Gemini** · `gemini-flash-lite-latest` | `GEMINI_API_KEY` | same |
| 🖥️ **Ollama** · `qwen2.5:3b` | none | same, locally |

**🔑 How the key is read — the notebook will never ask for it.** There is **no dialog box**: the key is looked up in **Colab Secrets** (🔑 icon in the left sidebar), then in environment variables. Set `PROVIDER` in the connector cell, run it, and it will tell you plainly whether the model answers — with the steps to follow if the key is missing.

**🔍 What to notice:** the local planner is **deterministic** — it does not really understand the question, it matches it against the metadata. That is enough for everyday questions (“trend of X in country Y since 2015”) and lets the whole workshop run without a key. A real model additionally handles unexpected phrasings and compound questions.

> ⚠️ **Free does not mean unconditional.** Before sending real statistical data to an online service, read its terms of use. For confidential data, prefer Ollama.

In [12]:
# ── Step 8 · Local planner (no key, no network) ─────────────────────────────
class LocalPlanner:
    """Builds a plan by matching the question against the WHOLE catalog (all dataflows)."""
    provider, model = "demo", "local planner"

    def __init__(self, catalog, default_flow):
        self.catalog, self.default_flow = catalog, default_flow
        self.calls = self.retries = self.tokens = 0

    def __repr__(self):
        return "🧪 demo · local planner"

    @staticmethod
    def _simplify(label):
        return re.sub(r"[^a-z0-9 ]", " ", re.sub(r"\([^)]*\)", " ", _norm(label))).strip()

    # unit and formatting words: they must not prevent a match
    UNITS = {"milliers", "millions", "milliards", "tonnes", "usd", "courants", "constants", "total", "totale",
              "pourcentage", "pct", "habitant", "habitants", "annuelle", "annuel", "adultes", "naissance",
              "reel", "reelle", "nominal", "indice", "prix", "consommation", "ratio"}

    @classmethod
    def _words(cls, text, retirer_unites=False):
        stop_words = {"the", "a", "an", "of", "to", "in", "on", "for", "and", "or", "is", "are", "with", "from", "by",
                 "rate", "which", "what", "how", "many", "since", "between", "trend", "compare", "evolution"}
        words = {m for m in re.findall(r"[a-z0-9]+", _norm(text)) if len(m) > 2 and m not in stop_words}
        return words - cls.UNITS if retirer_unites else words

    def _choose_flow(self, question):
        """Selects the most relevant dataflow in the whole catalogue."""
        results = self.catalog.search_flows(question, k=1)
        return results.iloc[0]["flow"] if not results.empty else self.default_flow

    def _detected_codes(self, question, flow):
        """Spots, inside the chosen dataflow, the codes mentioned in the question."""
        structure = self.catalog.structure(flow)
        q, question_words = _norm(question), self._words(question)
        found = {}
        for dim in structure["dimensions"]:
            selection = []
            for code in dim["codes"]:
                simple = self._simplify(code["label"])
                code_words = self._words(simple, retirer_unites=True)      # “thousand tonnes” is not discriminating
                if simple and simple in q:
                    selection.append(code["code"])
                elif re.search(rf"\b{re.escape(_norm(code['code']))}\b", q):
                    selection.append(code["code"])
                elif code_words and code_words <= question_words:        # every useful word of the code is in the question
                    selection.append(code["code"])
                elif code_words and len(code_words & question_words) >= max(2, len(code_words) - 1):
                    selection.append(code["code"])
            if selection:
                found[dim["id"]] = sorted(set(selection))[:4]
        return structure, found

    def __call__(self, instructions, messages, json_strict=False):
        self.calls += 1
        question = messages[0]["content"]
        if instructions.startswith("You write"):
            return self._summarise(messages[-1]["content"])
        if len(messages) > 1:
            return json.dumps({"give_up": "the local planner has no alternative to offer"}, ensure_ascii=False)

        q = _norm(question)
        flow = self._choose_flow(question)
        structure, codes = self._detected_codes(question, flow)
        years = re.findall(r"\b((?:19|20)\d{2})\b", question)
        order = [d["id"] for d in structure["dimensions"]]
        key = ".".join("+".join(codes.get(d, [])) for d in order)

        plan = [{"goal": f"identify the right dataset ({flow})", "tool": "search_dataset",
                 "args": {"question": question[:80]}}]
        if not any(codes.values()):
            plan.append({"goal": "inspect the dimensions of the chosen dataflow", "tool": "describe_dataflow",
                         "args": {"flow": flow}})
        plan.append({"goal": "fetch the observations", "tool": "fetch_data",
                     "args": {"flow": flow, "key": key if key.strip(".") else "all",
                              "start": years[0] if years else None,
                              "end": years[-1] if len(years) > 1 else None,
                              "last_n": None if years else 10}})
        if any(mot in q for mot in ("evolution", "tendance", "depuis", "compare", "comparer", "graphique", "courbe")):
            plan.append({"goal": "plot the series", "tool": "plot_chart",
                         "args": {"dataset": "{{dataset}}", "title": question[:60]}})
        operation = ("change" if any(m in q for m in ("change", "evolution", "depuis", "progression"))
                     else "ranking" if any(m in q for m in ("ranking", "which country", "highest", "compare"))
                     else "latest_value")
        plan.append({"goal": f"compute: {operation}", "tool": "analyse_dataset",
                     "args": {"dataset": "{{dataset}}", "operation": operation}})
        if any(mot in q for mot in ("export", "csv", "enregistre", "telecharge", "sauvegarde")):
            plan.append({"goal": "export the table", "tool": "export_csv",
                         "args": {"dataset": "{{dataset}}", "file_name": "extraction.csv"}})
        return json.dumps({"plan": [{"goal": p["goal"], "tool": p["tool"],
                                     "args": {k: v for k, v in p["args"].items() if v is not None}} for p in plan]},
                          ensure_ascii=False)

    @staticmethod
    def _summarise(contenu):
        """Writes a short answer from tool results only."""
        results = contenu.split("RESULTS:", 1)[-1].strip()
        blocks = {b.split("]")[0].lstrip("["): b.split("]", 1)[-1].strip()
                 for b in results.split("\n[") if "]" in b}
        rows = []
        extraction = blocks.get("fetch_data", "")
        if extraction.startswith("dataset#"):
            summary = extraction.split(" : ", 1)[-1].split(" · columns")[0]
            rows.append(f"I queried the service and retrieved **{summary}**.")
        analysis = blocks.get("analyse_dataset", "")
        if analysis:
            header, *body = analysis.splitlines()
            operation = header.split(":", 1)[-1].split(" sur ")[0].strip()
            rows.append(f"Result of the requested computation ({operation}) :")
            rows += [f"- {l.strip()}" for l in body[1:7] if l.strip()]
        if "join_datasets" in blocks:
            rows.append("The two datasets were combined on country and year.")
        if "plot_chart" in blocks:
            rows.append("The chart above shows exactly these observations.")
        if "waiting" in blocks.get("export_csv", ""):
            rows.append("📝 The CSV export was **queued for approval**: no file is written without validation.")
        if not rows:
            rows.append("The calls produced no usable result. Rephrase, giving the country, "
                          "the indicator and the period.")
        return "\n".join(rows)


LOCAL_BRAIN = LocalPlanner(CATALOG, DEFAULT_FLOW)
callout("The local planner is ready: it picks the dataflow from <b>the whole catalogue</b>, then spots the codes — with no extra network call.",
        "Demonstration brain", "success")

In [13]:
# ── Step 8 (cont.) · Connector to free models ───────────────────────────────
# ⚙️ Brain settings: these two variables ARE actually used by the interface (step 10).
PROVIDER = "auto"      # "local" (no key) · "auto" (first available provider) · "groq" · "gemini" · "ollama"
MODEL = None             # None = the provider's default model; e.g. "openai/gpt-oss-120b"

# 🆓 FREE models only: any other value is refused by the connector.
# Source: console.groq.com/docs/models and /docs/rate-limits (checked September 2026).
# ⚠️ llama-3.1-8b-instant and llama-3.3-70b-versatile were retired by Groq on 2026-08-16.
FREE_MODELS = {
    "groq": {"openai/gpt-oss-20b": "30 req/min · ~1,000 req/day · fast, fine for simple JSON",
             "openai/gpt-oss-120b": "30 req/min · ~1,000 req/day · better reasoning",
             "qwen/qwen3.8-27b": "30 req/min · ~1,000 req/day · multilingual, JSON mode"},
    "gemini": {"gemini-flash-lite-latest": "Google AI Studio free tier · a few hundred req/day",
               "gemini-flash-latest": "Google AI Studio free tier · lower quota"},
    "ollama": {"qwen2.5:3b": "local, unlimited, offline", "llama3.2:3b": "local, unlimited, offline",
               "phi3.5:3.8b": "local, unlimited, offline"},
}

PROVIDERS = {
    "groq":   {"url": "https://api.groq.com/openai/v1", "key": "GROQ_API_KEY",
               "modele": "openai/gpt-oss-20b", "appels_par_minute": 30},
    "gemini": {"url": "https://generativelanguage.googleapis.com/v1beta/openai", "key": "GEMINI_API_KEY",
               "modele": "gemini-flash-lite-latest", "appels_par_minute": 10},
    "ollama": {"url": os.environ.get("OLLAMA_URL", "http://localhost:11434") + "/v1", "key": None,
               "modele": "qwen2.5:3b", "appels_par_minute": 600},
}

def read_secret(name):
    if not name:
        return None
    value = os.environ.get(name)
    if not value:
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:
            value = None
    return value

def ollama_available():
    try:
        return requests.get(PROVIDERS["ollama"]["url"].removesuffix("/v1") + "/api/tags", timeout=2).ok
    except Exception:
        return False


class ProviderError(Exception):
    pass


class OpenAICompatibleModel:
    """Adapter (instructions, messages) -> text for any /chat/completions API."""

    def __init__(self, provider="auto", modele=None, temperature=0.0, max_tokens=900, max_reprises=3):
        self.provider = self._choose(provider)
        conf = PROVIDERS[self.provider]
        self.url = conf["url"].rstrip("/") + "/chat/completions"
        self.key = read_secret(conf["key"]) if conf["key"] else "ollama"
        self.modele = modele or conf["modele"]
        if self.modele not in FREE_MODELS.get(self.provider, {}) and self.provider != "ollama":
            allowed_models = ", ".join(FREE_MODELS.get(self.provider, {}))
            raise ProviderError(
                f"“{self.modele}” is not in the free-model list for {self.provider}. "
                f"Allowed models: {allowed_models}")
        self.interval = 60.0 / conf["appels_par_minute"] * 1.05
        self.temperature, self.max_tokens, self.max_reprises = temperature, max_tokens, max_reprises
        self.calls, self.retries, self.tokens, self._dernier = 0, 0, 0, 0.0

    @staticmethod
    def _choose(provider):
        if provider != "auto":
            conf = PROVIDERS[provider]
            if conf["key"] and not read_secret(conf["key"]):
                raise ProviderError(f"key {conf['key']} missing")
            if provider == "ollama" and not ollama_available():
                raise ProviderError("Ollama server unreachable")
            return provider
        for name in ("groq", "gemini"):
            if read_secret(PROVIDERS[name]["key"]):
                return name
        if ollama_available():
            return "ollama"
        raise ProviderError("no provider: add GROQ_API_KEY or GEMINI_API_KEY, or start Ollama")

    def __call__(self, instructions, messages, json_strict=False):
        body = {"model": self.modele, "temperature": self.temperature, "max_tokens": self.max_tokens,
                 "messages": [{"role": "system", "content": instructions}] + messages}
        if json_strict and self.provider in ("groq", "gemini", "ollama"):
            body["response_format"] = {"type": "json_object"}          # the model CANNOT reply with anything else
        if self.provider == "groq" and self.modele.startswith("openai/gpt-oss"):
            body["reasoning_effort"] = "low"                           # leaves tokens for the answer itself
        for attempt in range(self.max_reprises + 1):
            waiting = self.interval - (time.monotonic() - self._dernier)
            if waiting > 0:
                time.sleep(waiting)
            self._dernier = time.monotonic()
            self.calls += 1
            r = requests.post(self.url, json=body, timeout=120,
                              headers={"Authorization": f"Bearer {self.key}", "Content-Type": "application/json"})
            if r.status_code in (429, 500, 502, 503) and attempt < self.max_reprises:
                timeout = min(60, float(r.headers.get("retry-after") or 0) or 2 ** (attempt + 1))
                self.retries += 1
                print(f"     ⏳ HTTP {r.status_code} → retrying in {timeout:.0f} s")
                time.sleep(timeout)
                continue
            if not r.ok:
                if r.status_code == 400 and "response_format" in body:
                    body.pop("response_format")                        # service without JSON mode: retry without it
                    continue
                raise ProviderError(f"HTTP {r.status_code} : {r.text[:200]}")
            data = r.json()
            self.tokens += (data.get("usage") or {}).get("total_tokens", 0) or 0
            choix = data["choices"][0]
            message = choix.get("message", {})
            text = (message.get("content") or "").strip()
            if not text:                                               # “reasoning” models: fall back on the reasoning field
                text = (message.get("reasoning") or message.get("reasoning_content") or "").strip()
            if choix.get("finish_reason") == "length" and body["max_tokens"] < 3000:
                body["max_tokens"] *= 2                                # truncated reply: retry with a bigger budget
                self.retries += 1
                continue
            if text:
                return text
            if body.get("reasoning_effort") == "low":
                body["reasoning_effort"] = "medium"                     # last resort
                continue
            raise ProviderError("the model returned an empty reply "
                                    f"(finish_reason={choix.get('finish_reason')})")
        raise ProviderError("quota still exhausted after several attempts")

    def __repr__(self):
        return f"{self.provider} · {self.modele}"


show_table(pd.DataFrame([{"provider": f, "free model": m, "quota / note": q}
                      for f, models in FREE_MODELS.items() for m, q in models.items()]),
        "Only these models are accepted: any other value of MODEL is refused before a single call.")
show_table(pd.DataFrame([{"provider": n, "key variable": c["key"] or "— (local)",
                       "key detected": "✅" if (read_secret(c["key"]) if c["key"] else ollama_available()) else "—",
                       "default model": c["modele"]} for n, c in PROVIDERS.items()]),
        "The notebook never asks for your key: it reads it from Colab Secrets or the environment.")


KEY_HELP = ("To use a model: open the <b>🔑 Secrets</b> icon in Colab's left sidebar → "
            "<b>Add new secret</b> → name <code>GROQ_API_KEY</code> (free key at "
            "<code>console.groq.com</code>) → paste the value → switch on <b>Notebook access</b>, "
            "then re-run this cell.<br><br><b>Prefer to paste the key manually?</b> Run <code>LLM_BRAIN = enter_key()</code> in a new cell — input is masked and the key is kept for this session only.")


def prepare_brain(provider=PROVIDER, modele=MODEL, tester=True):
    """Creates the requested brain and CHECKS that it answers. Returns None if no model is available."""
    if provider == "local":
        callout("Setting <code>PROVIDER = \"local\"</code>: the local planner will be used.", "Brain", "info")
        return None
    try:
        brain = OpenAICompatibleModel(provider, modele)
    except ProviderError as e:
        callout(f"<b>{e}</b><br>{KEY_HELP}<br><br>In the meantime the agent runs on the "
                "<b>local planner</b>: the whole notebook stays usable.", "No model available", "warning")
        return None
    if not tester:
        return brain
    try:
        answer = brain("Reply with the single word OK.", [{"role": "user", "content": "connection test"}])
        callout(f"Active brain: <b>{brain!r}</b> · test reply: <code>{html.escape(answer.strip()[:40])}</code>"
                f"<br>The interface (step 10) will offer it by default.", "Model connected", "success")
        return brain
    except Exception as e:
        callout(f"The key was found, but the call failed: <b>{type(e).__name__} — {e}</b>.<br>"
                "Check the model name, or switch provider. The local planner takes over.",
                "Call refused", "danger")
        return None


ASK_FOR_KEY = True     # allows manual key entry (masked, never stored in the notebook)

def enter_key(name="GROQ_API_KEY", provider=None, modele=None):
    """Manual key entry, if you would rather not use Colab Secrets.
    Usage:  LLM_BRAIN = enter_key()        ← the key is kept for this session only."""
    import getpass
    if not ASK_FOR_KEY:
        callout("Manual entry is disabled (<code>ASK_FOR_KEY = False</code>).", "Entry refused", "warning")
        return None
    value = getpass.getpass(f"Paste your {name} key and press Enter (input stays hidden): ").strip()
    if not value:
        callout("No key entered.", "Cancelled", "warning")
        return None
    os.environ[name] = value
    brain = prepare_brain(provider or {"GROQ_API_KEY": "groq", "GEMINI_API_KEY": "gemini"}.get(name, "auto"), modele)
    globals()["LLM_BRAIN"] = brain
    return brain


def test_planning(brain=None, question="Trend of the first catalogue indicator for one country since 2018"):
    """Diagnostic: shows the RAW plan produced by the model, useful when planning fails."""
    brain = brain or LLM_BRAIN or LOCAL_BRAIN
    try:
        raw = brain(plan_instructions(TOOLS, STRUCTURE, SERVICE), [{"role": "user", "content": question}], json_strict=True)
    except TypeError:
        raw = brain(plan_instructions(TOOLS, STRUCTURE, SERVICE), [{"role": "user", "content": question}])
    except Exception as e:
        callout(f"Call impossible: <b>{type(e).__name__} — {e}</b>", "Diagnostic", "danger")
        return None
    try:
        plan = parse_plan(raw)
        callout("<br>".join(f"{e['number']}. <code>{e['tool']}</code> — {html.escape(e['goal'])}" for e in plan),
                f"Readable plan ({brain!r})", "success")
    except Exception as e:
        callout(f"The plan could not be read: <b>{e}</b><br>Raw reply:<br>"
                f"<pre style='font-size:11.5px;white-space:pre-wrap'>{html.escape(raw[:600]) or '(vide)'}</pre>",
                "Unusable format", "warning")
    return raw


LLM_BRAIN = prepare_brain()

provider,free model,quota / note
groq,openai/gpt-oss-20b,"30 req/min · ~1,000 req/day · fast, fine for simple JSON"
groq,openai/gpt-oss-120b,"30 req/min · ~1,000 req/day · better reasoning"
groq,qwen/qwen3.8-27b,"30 req/min · ~1,000 req/day · multilingual, JSON mode"
gemini,gemini-flash-lite-latest,Google AI Studio free tier · a few hundred req/day
gemini,gemini-flash-latest,Google AI Studio free tier · lower quota
ollama,qwen2.5:3b,"local, unlimited, offline"
ollama,llama3.2:3b,"local, unlimited, offline"
ollama,phi3.5:3.8b,"local, unlimited, offline"


provider,key variable,key detected,default model
groq,GROQ_API_KEY,—,openai/gpt-oss-20b
gemini,GEMINI_API_KEY,—,gemini-flash-lite-latest
ollama,— (local),—,qwen2.5:3b


### 🔑 No key configured?

If the cell above shows **“No model available”**, that is expected: the notebook **never asks** for the key through a dialog box, because a key typed into a cell would be saved in plain text inside the `.ipynb` file and shared with it.

Two options:

| | Method | Advantage |
|:--|:--|:--|
| **A** | **🔑 Secrets** icon (left sidebar) → **Add new secret** → name exactly `GROQ_API_KEY` → paste the key from `console.groq.com` → switch on **Notebook access** → re-run the previous cell | survives session restarts |
| **B** | Uncomment and run the cell below | immediate; masked input; key valid for this session |

Either way you should then see **“✅ Model connected”**, and re-run the interface cell (step 10).

In [14]:
# ── Manual key entry (optional) ─────────────────────────────────────────────
# Uncomment ONE of the two lines, run the cell, then paste your key at the prompt.
# Input is masked and the key is NEVER written into the notebook.

# LLM_BRAIN = enter_key()                          # GROQ_API_KEY (default)
# LLM_BRAIN = enter_key("GEMINI_API_KEY", "gemini")

if LLM_BRAIN is None:
    callout("No active model: the agent will use the <b>local planner</b>, which works without a key. "
            "Uncomment a line above to plug in a free model.", "Current state", "info")
else:
    callout(f"Active model: <b>{LLM_BRAIN!r}</b> — the interface (step 10) will open on it.",
            "Current state", "success")

---
## Step 9 · Text-mode trial

**🎯 Goal:** validate the whole chain — plan, requests, computation, chart — before opening the interface.

The `ask()` function shows exactly what the interface will show: the plan, the call trace, the data table, the chart and the written answer.

In [15]:
# ── Step 9 · Answer formatting ──────────────────────────────────────────────
ICONS = {"lister_dataflux": "📚", "describe_dataflow": "🧱", "search_code": "🔤",
          "fetch_data": "📈", "analyse_dataset": "🧮", "plot_chart": "📊", "export_csv": "💾"}

def _chip(source):
    return (f"<span style='display:inline-block;background:#E8F5EF;color:#0F5A31;border:1px solid #BFE3CE;"
            f"border-radius:12px;padding:0 8px;margin:0 2px;font-size:11.5px'>{ICONS.get(source, '📄')} {source}</span>")

def text_to_html(text):
    rows, puces = [], []
    def clear():
        if puces:
            rows.append("<ul style='margin:4px 0 4px 18px;padding:0'>" + "".join(f"<li>{p}</li>" for p in puces) + "</ul>")
            puces.clear()
    for brute in str(text).strip().splitlines():
        row = html.escape(brute.strip())
        row = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", row)
        row = re.sub(r"`([^`]+)`", r"<code style='background:#EEF2F0;padding:1px 5px;border-radius:4px'>\1</code>", row)
        row = re.sub(r"\[([A-Za-z0-9_,.\- ]{3,90})\]", lambda m: "".join(_chip(s.strip()) for s in m.group(1).split(",")), row)
        if re.match(r"^[-*•]\s+", brute.strip()):
            puces.append(re.sub(r"^[-*•]\s+", "", row)); continue
        clear()
        rows.append(row or "<div style='height:6px'></div>")
    clear()
    sortie = ""
    for i, l in enumerate(rows):
        bloc = l.startswith("<ul")
        sortie += ("" if i == 0 or bloc or rows[i - 1].startswith("<ul") else "<br>") + l
    return sortie

def user_bubble(question):
    return (f"<div style='display:flex;justify-content:flex-end;margin:10px 0'>"
            f"<div style='max-width:78%;background:#0B2545;color:#fff;padding:10px 14px;border-radius:14px 14px 2px 14px;"
            f"{FONT};font-size:14px'>🧑‍💼 {html.escape(question)}</div></div>")

def plan_to_html(execution):
    if not execution.plan:
        return "<i>none plan</i>"
    done = {e.number: e for e in execution.steps}
    rows = []
    for step in execution.plan:
        done_step = done.get(step["number"])
        status = "⏳" if done_step is None else ("✅" if done_step.result and not done_step.error else
                                            ("🕒" if done_step.reason.startswith("mise en file") else "⚠️"))
        rows.append(f"<li style='margin:2px 0'>{status} <b>{html.escape(step['tool'])}</b> — "
                      f"{html.escape(step['goal'])} <code style='font-size:11px'>"
                      f"{html.escape(json.dumps(step['args'], ensure_ascii=False)[:70])}</code></li>")
    extra = ("" if execution.corrections == 0 else
                  f"<div style='color:{GOLD};font-size:12px;margin-top:4px'>↻ {execution.corrections} fix(es) applied during execution</div>")
    return f"<ol style='margin:4px 0 0 18px;padding:0;{FONT};font-size:12.5px'>{''.join(rows)}</ol>{extra}"

def figure_to_html(execution, store=STORE):
    for dataset in reversed(execution.datasets):
        if dataset in store.figures:
            return (f"<img src='data:image/png;base64,{store.figures[dataset]}' "
                    f"style='width:100%;max-width:760px;border:1px solid #DDE3E0;border-radius:10px;margin-top:6px'/>")
    return ""

def data_to_html(execution, store=STORE, max_rows=8):
    if not execution.datasets:
        return ""
    dataset = execution.datasets[-1]
    table = store.table(dataset)
    columns = [c for c in table.columns if not c.endswith("_LIBELLE")]
    return (f"<div style='{FONT};font-size:12px;color:{GREY};margin-top:6px'>{dataset} · {len(table)} row(s)</div>"
            + df_to_html(table[columns], max_rows))

def agent_bubble(execution, brain):
    colour = GREEN if execution.answer and not execution.stop.startswith(("planning", "gave up")) else RED
    meta = (f"Agent SDMX · {brain!r} · {len(execution.steps)} step(s) · {execution.model_calls} model call(s) · "
            f"{execution.http_requests} HTTP request(s) · stop: {execution.stop}")
    details = (f"<div style='{FONT};font-size:12.5px;margin-bottom:3px'><b>🗺️ Plan</b></div>{plan_to_html(execution)}"
               f"<div style='{FONT};font-size:12.5px;margin:8px 0 3px'><b>🧭 Execution trace</b></div>"
               f"{df_to_html(execution.log(), 10)}")
    return (f"<div style='display:flex;justify-content:flex-start;margin:10px 0'>"
            f"<div style='max-width:92%;background:#fff;border:1px solid #DDE3E0;border-left:5px solid {colour};"
            f"padding:10px 14px;border-radius:14px 14px 14px 2px;{FONT};font-size:14px;line-height:1.6;"
            f"box-shadow:0 1px 3px rgba(0,0,0,.06)'>"
            f"<div>🤖 {text_to_html(execution.answer)}</div>"
            f"{figure_to_html(execution)}{data_to_html(execution)}"
            f"<div style='margin-top:8px;color:{GREY};font-size:11.5px'>{meta}</div>"
            f"<details style='margin-top:6px'><summary style='cursor:pointer;color:#12507A;font-size:12.5px'>"
            f"🗺️ Show the plan and trace</summary><div style='margin-top:6px'>{details}</div></details></div></div>")


def ask(question, brain=None, max_steps=8, verbose=False):
    brain = brain or LOCAL_BRAIN
    execution = SDMXAgent(TOOLS, brain, STRUCTURE, SERVICE, QUEUE, max_steps=max_steps).run_tool(question, verbose=verbose)
    display(HTML(user_bubble(question) + agent_bubble(execution, brain)))
    return execution

EXAMPLE = (f"Trend of {STRUCTURE['dimensions'][-1]['codes'][0]['label']} "
           f"for {STRUCTURE['dimensions'][1]['codes'][0]['label']} since 2018"
           if len(STRUCTURE["dimensions"]) > 2 and STRUCTURE["dimensions"][-1]["codes"] else "Give me the latest observations")
DEMO_RUN = ask(EXAMPLE, brain=LLM_BRAIN or LOCAL_BRAIN)

In [16]:
# ── Step 9 (cont.) · Self-test of the whole chain ───────────────────────────
test_questions = [EXAMPLE, "What are the latest available values?", "Compare the available countries"]
rows = []
for q in test_questions:
    ex = SDMXAgent(TOOLS, LOCAL_BRAIN, STRUCTURE, SERVICE, ApprovalQueue(TOOLS_BY_NAME)).run_tool(q)
    rows.append({"question": q[:52], "plan steps": len(ex.plan), "executed steps": len(ex.steps),
                   "datasets produced": len(ex.datasets), "HTTP requests": ex.http_requests,
                   "answer": "✅" if ex.answer else "❌", "stop": ex.stop})
show_table(pd.DataFrame(rows), "The whole chain is exercised: planning, SDMX calls, computation, rendering.")
# Demonstration: combining TWO different dataflows on country and year
try:
    country = next((c["code"] for d in STRUCTURE["dimensions"] if d["id"].upper().startswith("REF") for c in d["codes"]), None)
    dataset_a, dataset_b = list(CATALOG.flow["flow"])[:2]
    ind_a = CATALOG.index[(CATALOG.index["flow"] == dataset_a) & (CATALOG.index["dimension"] == "INDICATOR")]
    ind_b = CATALOG.index[(CATALOG.index["flow"] == dataset_b) & (CATALOG.index["dimension"] == "INDICATOR")]
    if country and not ind_a.empty and not ind_b.empty:
        tool = TOOLS_BY_NAME["fetch_data"]
        ja = tool.run_tool(flow=dataset_a, key=f".{country}.{ind_a.iloc[0]['code']}", last_n=8).split(" ")[0]
        jb = tool.run_tool(flow=dataset_b, key=f".{country}.{ind_b.iloc[0]['code']}", last_n=8).split(" ")[0]
        result = TOOLS_BY_NAME["join_datasets"].run_tool(jeu_a=ja, jeu_b=jb)
        callout(f"<code>{dataset_a}</code> × <code>{dataset_b}</code> combined on REF_AREA and TIME_PERIOD:<br>"
                f"<pre style='font-size:11.5px;margin:4px 0'>{html.escape(result[:450])}</pre>",
                "Combining two datasets", "success")
except Exception as e:
    callout(f"Combination not demonstrated here ({type(e).__name__} : {e}).", "Combination", "warning")

show_table(pd.DataFrame(CLIENT.http_log[-8:]), "Latest HTTP requests actually sent to the service "
        f"({CLIENT.requests_count} request(s), {CLIENT.from_cache} reply(ies) served from cache).")

question,plan steps,executed steps,datasets produced,HTTP requests,answer,stop
Trend of Adult literacy rate for Cote d'Ivoire since,3,3,2,1,✅,plan completed
What are the latest available values?,4,4,2,1,✅,plan completed
Compare the available countries,5,5,2,1,✅,plan completed


url,status,ms
[demo] dataflow/DEMO/DF_POP_DEMO/latest,demo,0
[demo] dataflow/DEMO/DF_TRADE_DEMO/latest,demo,0
"[demo] data/DEMO,DF_EDU_DEMO,latest/.CIV.LITERACY",demo,0
"[demo] data/DEMO,DF_EDU_DEMO,latest/.CIV.LITERACY",demo,0
"[demo] data/DEMO,DF_EDU_DEMO,latest/all",demo,0
"[demo] data/DEMO,DF_EDU_DEMO,latest/all",demo,0
"[demo] data/DEMO,DF_EDU_DEMO,latest/.CIV.LITERACY",demo,0
"[demo] data/DEMO,DF_HEALTH_DEMO,latest/.CIV.LIFE_EXP",demo,0


---
# 🟥 PART 5 — The plain-language interface

## Step 10 · Ask a question, get a table and a chart

**🎯 Goal:** put the SDMX agent in the hands of a user who knows neither SDMX nor country codes.

| Area | Role |
|:--|:--|
| **⚙️ Settings** | brain (local planner or model), working dataflow, step budget |
| **💡 Suggestions** | template questions built from the **codes actually available** |
| **💬 Conversation** | the answer, the chart, the table, and the collapsible plan |
| **🗺️ Plan** | the plan of the latest question, step by step |
| **📚 Catalogue** | every published dataset, with a search box |
| **🌐 Requests** | the exact SDMX URLs, to reproduce the answer by hand |
| **✅ Approvals** | CSV exports awaiting validation |
| **🧾 Log** | hash-chained audit logs and their integrity |

### 🔬 Experiments to try

1. Ask a question **with a country and a period**; then open the **🌐 Requests** tab and paste the URL into your browser: you must find exactly the same figures.
2. Ask for a **CSV export**, then approve it in **✅ Approvals**.
3. Change the **dataflow** in the settings: tools, suggestions and codes are regenerated automatically.
4. With a Groq key, compare the **plan** written by the model with the local planner's.

In [ ]:
# ── Step 10 · The interface ─────────────────────────────────────────────────
import ipywidgets as widgets

BRAINS = [("🧪 Local planner (no key)", "local"), ("⚡ Groq — free", "groq"),
            ("🔷 Gemini — free", "gemini"), ("🖥️ Ollama — local", "ollama")]


def suggestions_from_catalog(catalog, structure, n=5):
    """Builds template questions from SEVERAL indexed dataflows."""
    proposals = []
    index = catalog.index
    if not index.empty:
        areas = index[index["dimension"].str.upper().str.startswith(("REF_AREA", "AREA", "COUNTRY", "GEO"))]
        country = list(areas["label"].unique())[:3]
        # drop non-informative dimensions: frequency, time, unit…
        useless = index["dimension"].str.upper().str.startswith(("FREQ", "TIME", "UNIT", "OBS", "DECIMAL"))
        others = index[~index.index.isin(areas.index) & ~useless]
        for flow, group in list(others.groupby("flow"))[:4]:
            labels = [l for l in group["label"].unique() if len(l) > 4][:1]
            if labels and country:
                proposals.append(f"Trend of {labels[0]} for {country[0]} since 2018")
        if len(country) > 1 and not others.empty:
            indicator = next((l for l in others["label"].unique() if len(l) > 4), None)
            if indicator:
                proposals.append(f"Compare {indicator} between {country[0]} and {country[1]}")
    proposals += ["Which datasets contain population data?",
                     "What are the latest available values?"]
    return proposals[:n]


class SDMXInterface:
    def __init__(self, client, structure, service, file):
        self.client, self.structure, self.service, self.file = client, structure, service, file
        self.catalog = CATALOG
        self.tools = TOOLS
        self.brains = {"local": LOCAL_BRAIN}
        if LLM_BRAIN is not None:                      # brain prepared and tested at step 8
            self.brains[LLM_BRAIN.provider] = LLM_BRAIN
        self.fil, self.executions, self.logs = [], [], []
        self.folder = OUTPUTS / "interface"
        self.folder.mkdir(parents=True, exist_ok=True)
        self._build()
        self._welcome()

    # ── construction ───────────────────────────────────────────────────────
    def _build(self):
        L = widgets.Layout
        header = widgets.HTML(
            f"<div style='background:linear-gradient(135deg,#0B2545 0%,#12507A 50%,#1B7A43 100%);border-radius:14px;"
            f"padding:14px 20px;{FONT}'><div style='color:#F2A900;font-size:11px;letter-spacing:2.5px;font-weight:700'>"
            f"SERVICE SDMX · {self.service['name'].upper()}</div>"
            f"<div style='color:#fff;font-size:21px;font-weight:800;margin-top:3px'>📊 Agent SDMX</div>"
            f"<div style='color:#d6e4ef;font-size:13px'>Ask your question in plain English. The agent plans, "
            f"queries the service, computes, then shows the table and the chart.</div></div>")

        style = {"description_width": "80px"}
        default = LLM_BRAIN.provider if LLM_BRAIN is not None else "local"
        self.w_brain = widgets.Dropdown(options=BRAINS, value=default, description="🧠 Brain",
                                          style=style, layout=L(width="320px"))
        flow_options = [("🔎 automatic — the agent picks from the whole catalogueue", ("auto", "auto"))] + \
                       [(f"{r.flow} — {r.name[:40]}", (r.agency, r.flow)) for r in self.catalog.flow.itertuples()]
        self.w_flow = widgets.Dropdown(options=flow_options, value=("auto", "auto"),
                                       description="📚 Dataflow", style=style, layout=L(width="430px"))
        self.w_flow.observe(self._change_flow, names="value")
        self.w_budget = widgets.IntSlider(value=8, min=3, max=12, description="⏱️ Budget", style=style, layout=L(width="300px"))
        settings = widgets.VBox([widgets.HBox([self.w_brain, self.w_budget]), self.w_flow])

        self.w_suggestions = widgets.HBox(layout=L(flex_flow="row wrap"))
        self._fill_suggestions()

        self.w_chat = widgets.HTML(layout=L(width="100%"))
        self.w_question = widgets.Text(placeholder="e.g. trend of inflation in Senegal since 2018…", layout=L(width="70%"))
        self.w_send = widgets.Button(description="Send", icon="paper-plane", button_style="success", layout=L(width="130px"))
        self.w_clear = widgets.Button(description="Clear", icon="trash", layout=L(width="110px"))
        self.w_send.on_click(self._send)
        self.w_clear.on_click(self._clear)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            try:
                self.w_question.on_submit(self._send)
            except Exception:
                pass
        self.w_status = widgets.HTML()

        self.w_plan = widgets.HTML()
        self.w_requests = widgets.HTML()
        self.w_approvals = widgets.VBox()
        self.w_log = widgets.HTML()
        self.w_catalog = widgets.HTML()
        self.w_search = widgets.Text(placeholder="Search an indicator or a dataset…", layout=L(width="60%"))
        self.w_search_btn = widgets.Button(description="Search", icon="search", layout=L(width="120px"))
        self.w_search_btn.on_click(self._search_catalog)
        catalog_tab = widgets.VBox([widgets.HBox([self.w_search, self.w_search_btn]), self.w_catalog])
        self.tabs = widgets.Tab(children=[self.w_plan, catalog_tab, self.w_requests,
                                             self.w_approvals, self.w_log])
        for i, title in enumerate(["🗺️ Plan", "📚 Catalogue", "🌐 Requests", "✅ Approvals", "🧾 Log"]):
            self.tabs.set_title(i, title)

        def title(t):
            return widgets.HTML(f"<div style='{FONT};font-weight:700;color:#0B2545;margin:10px 0 2px'>{t}</div>")

        self.app = widgets.VBox([header, title("⚙️ Settings"), settings, title("💡 Suggestions"), self.w_suggestions,
                                 title("💬 Conversation"), self.w_chat,
                                 widgets.HBox([self.w_question, self.w_send, self.w_clear]), self.w_status,
                                 title("🔬 Traceability"), self.tabs], layout=L(width="100%", max_width="1020px"))

    def _fill_suggestions(self):
        buttons = []
        for s in suggestions_from_catalog(self.catalog, self.structure):
            b = widgets.Button(description=s if len(s) <= 48 else s[:46] + "…", tooltip=s,
                               layout=widgets.Layout(width="auto", margin="2px"))
            b.on_click(lambda _, q=s: sstatetr(self.w_question, "value", q))
            buttons.append(b)
        self.w_suggestions.children = buttons

    # ── affichage ──────────────────────────────────────────────────────────
    def _welcome(self):
        dims = ", ".join(d["id"] for d in self.structure["dimensions"])
        self.fil = [f"<div style='{FONT};background:#fff;border:1px solid #DDE3E0;border-left:5px solid {GREEN};"
                    f"border-radius:14px;padding:10px 14px;font-size:14px'>🤖 Hello! I query the "
                    f"<b>{html.escape(self.service['name'])}</b> service, which publishes "
                    f"<b>{len(self.catalog.flow)}</b> dataset(s).<br>"
                    f"In <b>🔎 automatic</b> mode I pick the dataflow that fits your question myself; "
                    f"you can also force one in the settings.<br>"
                    f"Starting dataflow: <code>{self.structure['flow']}</code> — dimensions <code>{dims}</code>.<br>"
                    f"Ask your question, or pick a suggestion.</div>"]
        self._refresh()

    def _refresh(self):
        self.w_chat.value = (f"<div style='background:#F7FAF8;border:1px solid #DDE3E0;border-radius:12px;padding:6px 12px;"
                             f"height:470px;overflow-y:auto;{FONT}'>" + "".join(self.fil) + "</div>")
        self._refresh_requests(); self._refresh_approvals(); self._refresh_log()
        self._refresh_catalog()

    def _set_status(self, text, colour=GREY):
        self.w_status.value = f"<div style='{FONT};font-size:12.5px;color:{colour};margin:4px 2px'>{text}</div>"

    def _refresh_catalog(self, results=None):
        index = self.catalog.index
        header = (f"<div style='{FONT};font-size:12.5px;margin:4px 0'><b>{len(self.catalog.flow)}</b> dataset(s) published · "
                  f"<b>{index['flow'].nunique() if not index.empty else 0}</b> indexed · "
                  f"<b>{len(index)}</b> code(s) in the directory</div>")
        if results is not None:
            body = df_to_html(results, 15) if not results.empty else "<i>no result</i>"
        else:
            body = df_to_html(self.catalog.flow, 12)
        self.w_catalog.value = header + body

    def _search_catalog(self, _=None):
        text = self.w_search.value.strip()
        if not text:
            self._refresh_catalog()
            return
        flow = self.catalog.search_flows(text, k=8)
        codes = self.catalog.search_code(text)
        bloc = f"<div style='{FONT};font-size:12.5px;margin:6px 0'><b>Datasets</b></div>"
        bloc += df_to_html(flow[["flow", "name", "score"]], 8) if not flow.empty else "<i>none</i>"
        bloc += f"<div style='{FONT};font-size:12.5px;margin:8px 0 4px'><b>Matching codes</b></div>"
        bloc += df_to_html(codes[["flow", "dimension", "code", "label"]], 12) if not codes.empty else "<i>none</i>"
        self.w_catalog.value = bloc

    def _refresh_requests(self):
        log = self.client.http_log[-15:]
        if not log:
            self.w_requests.value = f"<div style='{FONT};font-size:13px;color:{GREY}'>No request yet.</div>"
            return
        rows = "".join(f"<li style='margin:3px 0;word-break:break-all'><code style='font-size:11.5px'>"
                         f"{html.escape(str(r['url']))}</code> <span style='color:{GREY}'>· {r['status']} · {r['ms']} ms</span></li>"
                         for r in log)
        self.w_requests.value = (f"<div style='{FONT};font-size:12.5px'>These URLs can be reproduced as-is "
                                 f"in a browser or with <code>curl</code>:<ul>{rows}</ul></div>")

    def _refresh_approvals(self):
        waiting = self.file.pending()
        blocks = [widgets.HTML(f"<div style='{FONT};font-size:13px'><b>{len(waiting)}</b> request(s) en waiting · "
                              f"{len(self.file.decisions)} decision(s)</div>")]
        for d in waiting:
            card = widgets.HTML(f"<div style='{FONT};border:1px solid {GOLD};background:#FFFBEF;border-radius:10px;"
                                 f"padding:8px 12px;font-size:12.5px'><b>Demande n°{d.number}</b> · <code>{d.tool}</code>"
                                 f"<br>{html.escape(json.dumps(d.args, ensure_ascii=False))}"
                                 f"<br><span style='color:{GREY}'>Question : {html.escape(d.task[:70])}</span></div>",
                                 layout=widgets.Layout(width="70%"))
            ok = widgets.Button(description="Approve", icon="check", button_style="success", layout=widgets.Layout(width="120px"))
            ko = widgets.Button(description="Reject", icon="times", button_style="danger", layout=widgets.Layout(width="110px"))
            ok.on_click(lambda _, n=d.number: self._decide(n, True))
            ko.on_click(lambda _, n=d.number: self._decide(n, False))
            blocks.append(widgets.HBox([card, widgets.VBox([ok, ko])]))
        if self.file.decisions:
            history = pd.DataFrame([{"#": e["number"], "tool": e["tool"], "decision": e["status"],
                                   "par": e["decided_by"], "empreinte": e["hash"][:12] + "…"} for e in self.file.decisions])
            blocks.append(widgets.HTML(df_to_html(history)))
        self.w_approvals.children = blocks

    def _refresh_log(self):
        if not self.logs:
            self.w_log.value = f"<div style='{FONT};font-size:13px;color:{GREY}'>No log recorded yet.</div>"
            return
        rows = []
        for path, question in self.logs[-10:]:
            ok, state = verify_log(path)
            rows.append({"file": path.name, "question": question[:50], "integrity": f"{'🔒' if ok else '🚩'} {state}"})
        self.w_log.value = (f"<div style='{FONT};font-size:12.5px'>Folder: <code>{self.folder}</code></div>"
                                + df_to_html(pd.DataFrame(rows)))

    # ── actions ────────────────────────────────────────────────────────────
    def _change_flow(self, evenement):
        agency, flow = evenement["new"]
        if flow == "auto":
            self.fil.append(f"<div style='{FONT};text-align:center;color:#5c4500;background:#FFF7E0;border:1px dashed {GOLD};"
                            f"border-radius:10px;padding:6px 12px;margin:8px auto;font-size:12.5px'>"
                            f"🔎 Automatic mode: the agent will pick from the "
                            f"<b>{len(self.catalog.flow)}</b> datasets in the catalogue.</div>")
            self._set_status("🔎 Automatic dataflow selection enabled", GREEN)
            self._refresh()
            return
        self._set_status("⏳ Discovering the structure…", NAVY)
        try:
            self.structure = self.catalog.structure(flow, agency)
            self.tools = build_tools(self.catalog, flow)
            self.file = ApprovalQueue({o.name: o for o in self.tools})
            self.brains["local"] = LocalPlanner(self.catalog, flow)
            self._fill_suggestions()
            self.fil.append(f"<div style='{FONT};text-align:center;color:#5c4500;background:#FFF7E0;border:1px dashed {GOLD};"
                            f"border-radius:10px;padding:6px 12px;margin:8px auto;font-size:12.5px'>"
                            f"🔄 Dataflow changed: <b>{html.escape(self.structure['name'])}</b> — "
                            f"tools and suggestions regenerated</div>")
            self._set_status(f"✅ {self.structure['name']}", GREEN)
        except Exception as e:
            self._set_status(f"⚠️ {type(e).__name__} : {e}", RED)
        self._refresh()

    def _brain(self):
        key = self.w_brain.value
        if key not in self.brains:
            self.brains[key] = OpenAICompatibleModel(key, MODEL)        # the MODEL from step 8 is honoured
        return self.brains[key]

    def _send(self, _=None):
        question = self.w_question.value.strip()
        if not question:
            self._set_status("✍️ Type a question first.", GOLD)
            return
        self.w_question.value = ""
        self.w_send.disabled = True
        self.fil.append(user_bubble(question))
        self._refresh()
        self._set_status("⏳ Planning and querying the service…", NAVY)
        try:
            brain = self._brain()
            execution = SDMXAgent(self.tools, brain, self.structure, self.service, self.file,
                                  max_steps=self.w_budget.value).run_tool(question)
            self.executions.append(execution)
            self.fil.append(agent_bubble(execution, brain))
            self.w_plan.value = plan_to_html(execution) + df_to_html(execution.log(), 12)
            path = self.folder / f"journal_{len(self.logs) + 1:03d}.json"
            save_log(execution, path, {"brain": repr(brain), "flow": self.structure["flow"]})
            self.logs.append((path, question))
            if self.file.pending():
                self.fil.append(f"<div style='{FONT};text-align:center;background:#FFF7E0;border:1px dashed {GOLD};"
                                f"border-radius:10px;padding:6px 12px;margin:8px auto;font-size:12.5px;color:#5c4500'>"
                                f"🕒 Un export attend votre décision dans l’onglet <b>✅ Approbations</b>.</div>")
                self.tabs.selected_index = 2
            self._set_status(f"✅ {len(execution.steps)} step(s) · {execution.http_requests} HTTP request(s)", GREEN)
        except Exception as e:
            self.fil.append(f"<div style='{FONT};background:#FBECEA;border-left:5px solid {RED};border-radius:10px;"
                            f"padding:10px 14px;margin:8px 0;font-size:13.5px'>🤖 I could not process the request: "
                            f"<b>{type(e).__name__}</b> — {html.escape(str(e))}</div>")
            self._set_status(f"⚠️ {type(e).__name__}", RED)
        finally:
            self.w_send.disabled = False
            self._refresh()

    def _decide(self, number, approuver):
        request = self.file.decide(number, approuver)
        message = (f"✅ Request #{number} approved — {html.escape(request.result)}" if approuver
                   else f"⛔ Request #{number} rejected — no file written.")
        self.fil.append(f"<div style='{FONT};text-align:center;background:#F4F7F5;border:1px dashed "
                        f"{GREEN if approuver else RED};border-radius:10px;padding:6px 12px;margin:8px auto;"
                        f"font-size:12.5px'>{message}</div>")
        self._refresh()

    def _clear(self, _=None):
        self.executions.clear(); STORE.clear(); self._welcome()
        self._set_status("🧹 Conversation cleared (audit logs are kept).")

    def show(self):
        display(self.app)


APP = SDMXInterface(CLIENT, STRUCTURE, SERVICE, QUEUE)
APP.show()

## Step 11 · Under the hood

**🎯 Goal:** be able to explain, without the notebook in front of you, the journey of a question.

| # | Where | What happens |
|:--:|:--|:--|
| 1 | Interface | the question and the settings (brain, dataflow, budget) are read |
| 2 | Planner | one model call produces a **plan** of 1 to 6 steps, in JSON |
| 3 | Substitution | `{{dataset}}` and `{{code}}` are replaced by earlier results |
| 4 | Policy | request too broad → refused; CSV export → sent to the **approval queue** |
| 5 | SDMX client | the URL is built, cached, and **logged verbatim** |
| 6 | Store | the table is kept under an identifier; only a **summary** reaches the model |
| 7 | Fix | on failure, the error goes back to the model, which proposes a replacement step |
| 8 | Summary | a final call writes the answer **from tool results only** |
| 9 | Log | plan, steps, URLs and answer are recorded and chained with SHA-256 fingerprints |

**🧠 What sets this agent apart from the previous lab:** it does not read documents, it **queries a live service**. Two consequences: its answers change when the data change (hence the importance of logging the URL and the timestamp), and it can **put a production service under strain** if left unbridled (hence the request budgets and the observation limit).

---
# ⬛ PART 6 — Wrap up

## Step 12 · Limits, exercises and troubleshooting

### ⚠️ Limits of what you have built

- **The local planner is not a language model.** It matches labels to codes; it fails on unexpected phrasings. That is deliberate: it makes the workshop usable without a key.
- **Joining stays simple.** `join_datasets` aligns two extractions on country and year; heterogeneous dimensions (different units or frequencies) still need manual work.
- **Indexing is capped** by `MAX_FLOWS_INDEXED`: on a service publishing hundreds of dataflows, the ones left out are only found when the agent looks for them explicitly (set `INDEX_EVERYTHING = True` to index them all, at the cost of one request per dataflow).
- **No fine handling of SDMX attributes** (observation status, unit, multiplier). In production these must be read, because they change how a figure should be interpreted.
- **The cache never expires.** Handy in a workshop, dangerous in production: add a lifetime, or use `updatedAfter`.
- **No authentication.** Some instances require a token; the client would then need an `Authorization` header.

### ✅ Checkpoint

| Question | Answer |
|:--|:--|
| Why does the agent call `search_code` before `fetch_data`? | So it never invents a code: codes come from the service's codelists |
| Why does the data table not reach the model? | Cost and context: the model gets a summary, the data stay in the store |
| What does logging the URL guarantee? | **Reproducibility**: a third party can replay the request and check the figure |
| Which guard rails protect the service? | Observation limit, refusal of `all` without a period, request budget, cache |
| What happens when a step fails? | The error goes back to the model, which proposes a fix, up to three times |

### 🧑‍💻 Your turn

1. **Attributes.** Read `OBS_STATUS` and `UNIT_MEASURE`, and show the unit on the chart.
2. **Two dataflows.** Extend `join_datasets` to align different frequencies (annual vs quarterly).
3. **Freshness.** Add an `updatedAfter` parameter and a “Check for updates” button.
4. **Plan approval.** Have the user approve the **plan** before execution, for expensive requests.
5. **Richer export.** Produce an Excel workbook with the data, the metadata and the request URL.

### 🛟 Troubleshooting

| Symptom | Fix |
|:--|:--|
| “service unreachable” on the AfDB endpoint | The NSI service is probably internal only: run the notebook from the AfDB network, or stay on the fallback service |
| 404 on a data request | The key does not follow the dimension order: re-read `describe_dataflow` |
| 413 or an endless request | Narrow the key and the period; use `last_n` |
| The chart is empty | The dataset has no time dimension, or all values are missing |
| Answers frozen after a change on the service side | Clear the cache: delete the `stg17_sdmx/cache` folder |
| Planning fails | Run `test_planning()`: it shows the model's raw reply |
| The interface does not show | Re-run the step 10 cell, or use `ask("…")` |

---

*STG17 Workshop · AfDB / STATAFRIC · “SDMX Agent” · The notebook writes nothing without human approval and logs every request.*